# 05 — Seleção de variáveis
**Hackathon Databricks · Terra Signal · Grupo 4**

**Equipe:** André Ferreira, Antonio Menezes, Júlia Vale e Lucas Menghi

**O que este notebook faz:** decide **quais variáveis entram no modelo** e separa os clientes que ficam guardados para
a prova final.
1. **Carrega a gold e confere** se ela está como a modelagem espera (seção 1).
2. **Prepara as variáveis e separa treino e teste** (seção 2): retira as categóricas redundantes, cria as dummies e
   guarda 20% dos clientes para a avaliação final.
3. **Testes estatísticos e retirada de variáveis** (seção 3): correlação, qui-quadrado, Mann-Whitney com correção para
   múltiplos testes e busca de duplicatas exatas. Saem 12 variáveis; ficam **41**. Grava a separação e as listas para
   o 06.
4. **Conferência com os métodos da aula** (seções 4 e 5): seleção para frente por AUC, *stepwise* por AIC, BIC e
   p-valor, VIF e intervalos dos coeficientes, num modelo simples (regressão logística).
5. **As duas seleções se encontram?** (seção 6): confere se as variáveis escolhidas pelos métodos da aula estão entre
   as 41.

**Origem:** as seções 2 e 3 são o código do Antonio (notebook `modelo-antonio`), sem mudança. As seções 4 e 5 são a
seleção que fizemos com os métodos da disciplina de regressão, agora como **conferência**: dois caminhos diferentes
que chegam às mesmas variáveis dão mais confiança à escolha.

| # | Pergunta de negócio | Etapa |
|---|---|---|
| M1 | Que variáveis entram no modelo, e por quê? | 2. Preparação · 3. Testes e retirada · 6. Encontro das seleções |

**Etapas da escolha de variáveis:**

| Etapa | Onde | O que faz | Como olha as variáveis |
|---|---|---|---|
| **Triagem** | 04 | Dá uma nota a cada coluna: recomendada, candidata ou sugiro descartar, pelo IV e pela repetição | Uma de cada vez |
| **Seleção final** | **05, seções 2 e 3** | Testes estatísticos com correção para múltiplos testes e retirada das duplicatas exatas: 41 colunas | Uma de cada vez e em pares |
| **Conferência** | **05, seções 4 e 5** | Seleção para frente por AUC, *stepwise* e VIF, num modelo | Todas juntas, num modelo |
| **Modelagem** | 06 | Regularização (Lasso, Ridge, Elastic Net) e 6 algoritmos sobre as 41 colunas | O modelo que vai para a entrega |

| Notebook | Papel |
|---|---|
| 01 — Dicionário de dados | O que é cada coluna e o que está errado; grava a bronze |
| 02 — Preparação dos dados | Limpa e grava a silver |
| 03 — Análise exploratória | Descritiva, associações, churn, vazamento, veredito de cada variável |
| 04 — Variáveis derivadas e ABT | Derivadas, ABT e dicionário com a sugestão; grava a gold |
| **05 — Seleção de variáveis** | **Separa treino e teste e chega às 41 variáveis; confere com os métodos da aula** |
| 06 — Modelagem | Compara 6 modelos, escolhe o modelo e o limiar, publica as previsões e registra no MLflow |
| 07 — Segmentação | Agrupa os clientes por comportamento (K-means) e cruza com risco e valor |
| 08 — Entrega | Gera o `prediction` da entrega e o resumo para a apresentação |

In [ ]:
%pip install statsmodels

In [ ]:
dbutils.library.restartPython()

In [ ]:
%run ./_comum

## Glossário de siglas deste notebook

| Sigla | Por extenso | O que é, em uma frase | Onde entender melhor |
|---|---|---|---|
| **ABT** | *Analytical Base Table* (tabela analítica de base) | A tabela que o modelo lê: uma linha por cliente, uma coluna por variável | 04: seção 2 |
| **AUC** | *Area Under the Curve* (área sob a curva ROC) | A chance de o modelo dar nota de risco maior a um cliente que saiu do que a um que ficou (0,5 = sorteio; 1 = perfeito) | Este notebook: seção 5 |
| **ROC** | *Receiver Operating Characteristic* | A curva que mostra, para cada ponto de corte, quantos dos que saem o modelo pega e quantos alarmes falsos dá | Este notebook: seção 5 |
| **VIF** | *Variance Inflation Factor* (fator de inflação da variância) | O quanto uma variável é explicada pelas outras do modelo; acima de 5 a 10, há repetição demais (multicolinearidade) | Este notebook: seção 5.4 |
| **IV** | *Information Value* (valor da informação) | O quanto uma variável, sozinha, separa quem sai de quem fica | 03: Apêndice A.7 · 04: seção 1 |
| **L1 / Lasso** | Regularização L1 (*Least Absolute Shrinkage and Selection Operator*) | Uma forma de o próprio modelo zerar as variáveis que pouco ajudam | 06: seção 2 |
| **D1 a D25** | Decisões do checklist | Cada decisão do grupo, com o status (D20 a D23 são da segmentação, da modelagem e da entrega) | Checklist abaixo |
| **bronze, silver, gold** | Camadas da arquitetura medalhão | Dado bruto, limpo e pronto para o modelo | 01: seção 2 |
| **Holdout (base de teste)** | — | 20% dos clientes separados no início e usados só na avaliação final | Este notebook: 2.5 |
| **Pearson** | Correlação de Pearson | Força da relação linear entre duas variáveis numéricas (−1 a 1) | Este notebook: 3.2 e 3.3 |
| **Qui-quadrado, Fisher** | Testes de associação | Se uma variável 0/1 está associada ao churn; o de Fisher é o exato, para poucos casos | Este notebook: 3.4 |
| **Mann-Whitney** | Teste de Mann-Whitney | Se uma variável contínua difere entre quem sai e quem fica, sem supor distribuição normal | Este notebook: 3.4 |
| **V de Cramér** | — | Tamanho do efeito de uma associação entre categorias (0 = nenhuma, 1 = total) | Este notebook: 3.4 |
| **FDR (Benjamini-Hochberg)** | *False Discovery Rate* (taxa de falsas descobertas) | Correção do p-valor quando se fazem muitos testes, para não "achar" efeito por acaso | Este notebook: 3.4 |
| **AIC, BIC** | Critérios de informação de Akaike e Bayesiano | Medem o ajuste do modelo com uma penalidade por variável; menor é melhor | Este notebook: 5.5 |

## Checklist de decisões

**A mesma tabela em todos os notebooks.** ☑ = decidido (com a data); ◐ = sugestão nossa, a confirmar (as de variáveis, na modelagem); ☐ = em aberto.

| # | Decisão | Opções | Proposta | Impacto | Evidência no 01 | No 02 | Depende de | Decisão do grupo |
|---|---|---|---|---|---|---|---|---|
| D1 | Grafia de `PhoneService` (70 + 23) | Padronizar | Padronizar para `Yes`/`No` | Sem isso, `yes` e `Yes` viram categorias diferentes no modelo | 4.1 | Aplicado | Carlos: só erro de padronização (respondido) | ☑ Padronizar para `Yes`/`No` (10/11) |
| D2 | `TotalCharges` nulo (278 + 64) | a) preencher com `MonthlyCharges` × `tenure`; b) excluir; c) deixar ausente | a | Excluir não serve para os 64 da inferência, que precisam de previsão, e perde o sinal da 4.3. A regra vale para 99% dos clientes | 4.1, 4.2, 4.3 | Aplicado | G2; Carlos: preencher assim é "razoável, desde que lembrem que é uma aproximação" (respondido); falta saber por que o valor está ausente | ☑ Preencher com `MonthlyCharges` × `tenure`, como aproximação (10/11) |
| D3 | `TotalCharges` nulo com `tenure` = 0 (10 + 1) | a) 0; b) outro valor | a (sai da própria regra: mensalidade × 0) | Clientes novos, sem cobrança ainda; efeito pequeno | 4.2 | Aplicado | G2 | ☑ 0 (10/11) |
| D4 | `tenure` = `unknown` (19 + 4) | a) `TotalCharges` ÷ `MonthlyCharges`, arredondado; b) mediana; c) excluir | a; mediana do histórico no único cliente sem as duas colunas | `tenure` é das variáveis mais ligadas ao churn: um erro aqui muda a faixa de tempo de casa do cliente | 4.1, 4.2 | Aplicado | G2; Carlos: 72 meses é teto real (respondido) | ☑ `TotalCharges` ÷ `MonthlyCharges`, arredondado e limitado a 72; mediana do histórico no caso sem os dois (10/11) |
| D5 | Indicador `fl_total_charges_ausente` | Criar ou não | Criar e comparar o modelo com e sem ele | Preserva o sinal da ausência (33,1% contra 26,4% de churn, p = 0,016) | 4.3 | Criado | Carlos: por que falta `TotalCharges` | ☑ Criar; o modelo testa com e sem (10/11) |
| D6 | Indicador `fl_tenure_ausente` | Criar ou não | Criar, mas tende a sair na seleção (só 19 casos, p = 0,21) | Pouca informação; uma coluna a mais | 4.3 | Não criado | G2 | ☑ Não criar: 19 casos, p = 0,21 (10/11) |
| D7 | `TotalCharges` fora de ±20% da regra (47 + 9) | a) manter; b) investigar; c) excluir | a | Mostra que o preenchimento de D2 é uma aproximação | 4.2 | Mantidos | Carlos: o preço muda com plano e promoções, e não há histórico de preços (respondido) | ☑ Manter (10/11) |
| D8 | Quebras de linha no `CustomerFeedback` (85 + 15) | Remover ou manter | Remover, se o texto for usado | Só importa se o texto entrar na análise ou no modelo | 4.2 | Aplicado | D9 | ☑ Trocar a quebra de linha por espaço (10/11) |
| D9 | `CustomerFeedback` no modelo | a) com; b) sem; c) os dois | c, com o modelo sem texto como principal | Possível vazamento: uma regra sobre o texto acerta 98,9% | Célula 2 (seção 1); 7.3 do 03 | Fora do 02 (modelo) | G1; Carlos: feedback antes ou depois do cancelamento | ☑ Fora da ABT. O texto vai na tabela separada `gold_churn_texto`, para o modelo com texto (contraexemplo); o modelo oficial é sem texto (10/11) |
| D10 | Serviços com 3 categorias ou sim/não | a) manter 3; b) binário | Testar as duas | "No internet service" repete `InternetService` em 6 colunas: redundância | 3 | Fora do 02 (notebook 04) | G4 | ◐ Sugestão: serviços como sim/não (`_bin`), sempre junto com `InternetService`. A ABT traz as duas versões; seleção final na modelagem (10/11) |
| D11 | Fonte e schema do grupo | a) CSVs de `dados/`; b) tabelas `bronze_*` em `terra_signal` | b | Todos os notebooks; os números precisam bater entre os integrantes | 1.2, 2 | Aplicado: o 02 lê das tabelas `bronze_*` | G3; Carlos: CSV com quebra de linha, schema | ☑ Tabelas `bronze_*` em `workspace.terra_signal`; ignorar (sem apagar) as do upload em `workspace.default` (10/11) |
| D12 | `MonthlyIncome` no modelo | a) manter como candidata; b) ignorar | a, e deixar a seleção decidir | Base externa, sem garantia de precisão; os organizadores dizem que pode ser ignorada. Na exploratória, a renda não muda o risco | 3; seção 10 do 03 | Convertida para número e mantida | Carlos: renda familiar em R$, base externa (respondido) | ◐ Sugestão: descartar (IV abaixo de 0,02, base externa). Fica na ABT; seleção final na modelagem (10/11) |
| D13 | Linhas do `Prediction.csv` | a) 1.409 (clientes); b) 1.461 (linhas físicas) | a | Arquivo com o número errado de linhas pode invalidar a entrega | 1.2 | Fora do 02 (entrega) | Organização: confirmar | ☑ 1.409, uma por `customerID`; a confirmar com a organização (10/11) |
| D14 | Formato da gold | — | `gold_churn_treino` (5.634, com `Churn` 1/0) e `gold_churn_inferencia` (1.409, sem `Churn`), mesmas colunas e ordem; números como número, categorias como texto; comentário em cada coluna; `gold_churn_previsoes` depois do modelo | É o que a modelagem usa; formato estável evita retrabalho | — | Fora do 02 (notebook 04) | — | ☑ ABT com todas as candidatas (originais limpas, serviços nas duas versões, indicador e derivadas) + dicionário com a sugestão de cada coluna; texto em `gold_churn_texto` (10/11) |
| D15 | Outras variáveis sem poder preditivo | a) descartar já; b) deixar a seleção decidir | a: `gender`, `PhoneService` e `MultipleLines` | IV abaixo de 0,02; modelo mais simples de explicar; sexo é variável sensível | Seção 7 e 8 do 03 | Fora do 02 (notebook 04) | — | ◐ Sugestão: descartar `gender`, `PhoneService` e `MultipleLines` (IV abaixo de 0,02). Ficam na ABT; seleção final na modelagem (10/11) |
| D16 | `MonthlyCharges` e `TotalCharges` (repetem outras variáveis) | a) manter; b) trocar por derivada; c) deixar a seleção decidir | Decidir no 04, com as derivadas | Repetem `InternetService` e `tenure` (associação ≥ 0,8) | Seção 6 e 8 do 03 | Fora do 02 (notebook 04) | 04 | ◐ Sugestão: manter `MonthlyCharges` (ou a mensalidade relativa ao pacote) e descartar `TotalCharges` (repete `tenure`, ρ 0,89). As três ficam na ABT; seleção final na modelagem (10/11) |
| D17 | Fonte do 03 | a) CSVs; b) tabelas `bronze_*` | b | Coerência com a D11; os números não mudam | — | Fora do 02 (notebook 03) | — | ☑ Ler da bronze (10/11) |
| D18 | Divisão de papéis | — | A preparação entrega a ABT e sugere; a seleção final das variáveis é feita na modelagem | Quem decide a seleção final das variáveis | — | — | — | ☑ Como proposto (10/11) |
| D19 | Seleção de variáveis | a) só as recomendadas; b) recomendadas + seleção para frente por AUC; c) testes estatísticos com correção para múltiplos testes e retirada das duplicatas | c, conferida com b | É o que entra no modelo | — | — | Notebook 05 | ☑ As 41 colunas do notebook 05 (seções 2 e 3): saem 6 categóricas redundantes e 12 variáveis (duplicatas exatas, versões repetidas do tempo de casa, `gender` sem associação após a correção FDR). A conferência com os métodos da aula (11 variáveis, AUC 0,841) cabe dentro das 41 (11/11) |
| D20 | Segmentação de clientes | a) sem segmentação; b) K-means no comportamento, sem o churn, cruzado com risco × valor | b | Diz que tipo de cliente está em risco e que oferta faz sentido | — | — | Notebook 07 | ◐ Sugestão para o negócio e a apresentação (11/11) |
| D21 | Modelo final | a) logística simples; b) regularização (Lasso, Ridge, Elastic Net); c) árvores (*random forest*, XGBoost, CatBoost) | O de melhor desempenho e mais consistente entre treino, teste e base total | É o modelo que vai para a entrega | — | — | Notebook 06 | ☑ Notebook 06 (modelo do grupo): logística Lasso, 41 colunas, C = 0,3, classes balanceadas, vencedora pela regra de consistência (Ridge a 0,003); AUC 0,830 no teste. Atenção: o balanceamento infla a probabilidade (média de 41% para 27% de churn real) (11/11) |
| D22 | Ponto de corte do Sim/Não | a) 0,5; b) o que maximiza o F1; c) pelo custo da ação | b | Define quantos clientes viram "Yes" | — | — | Notebook 06 | ☑ 0,5046: maior F1 nas probabilidades fora da dobra do treino (notebook 06) (11/11) |
| D23 | Arquivo da entrega | — | 1.409 linhas, `customerID` e `prediction` (Yes/No), na ordem do `inference.csv`, no formato do notebook oficial | Arquivo fora do formato pode invalidar a entrega | — | — | D13; notebook 08 | ☑ `prediction_grupo4_<data_hora>.csv` gerado no 08 e conferido nas 8 regras: 571 de 1.409 com Yes (40,5%); falta comparar os nomes das colunas com o notebook oficial (11/11) |
| D24 | Monitoramento do modelo | a) sem monitoramento; b) PSI das variáveis e das notas, taxa de Yes e reprodução pelo MLflow, com semáforo e app | b | Avisa quando o modelo envelhece, antes de o resultado de negócio mostrar | — | — | Notebook 09 | ◐ Proposta: notebook 09 e app em Streamlit (11/11) |
| D25 | Motor de atendimento | a) só a fila de retenção; b) a nota, os motivos e a oferta de cada cliente na hora da ligação, e a comparação de pacotes para quem ainda não é cliente | b | Leva o modelo para quem fala com o cliente | — | — | Notebook 10 | ◐ Proposta: notebook 10 e página Atendimento do app (12/11) |

## 1. Carregar a gold e conferir

**O que faço:** leio as quatro tabelas gold gravadas pelo 04 e confiro o "contrato" da ABT (*Analytical Base Table*):
as regras que a modelagem precisa poder supor (seção 2 do 04).

**Por que um teste de carga:** a modelagem vai rodar em outros notebooks, possivelmente por outras pessoas e em
outro dia. Se a gold mudou (alguém rodou o 04 com outra versão, uma tabela não foi gravada, uma coluna sumiu), é
melhor descobrir **aqui**, com uma mensagem clara, do que com um modelo que treina sem erro e prevê errado.

In [ ]:
# 1.1 Carrega as quatro tabelas gold
assert "FQ" in globals(), "rode antes a célula %run ./_comum (ela define FQ, virg e o padrão dos gráficos)"
# ordeno por customerID: o Spark não garante a ordem das linhas, e a ordem muda quais clientes caem em cada dobra
treino = spark.table(f"{FQ}.gold_churn_treino").toPandas().sort_values("customerID").reset_index(drop=True)
inferencia = spark.table(f"{FQ}.gold_churn_inferencia").toPandas().sort_values("customerID").reset_index(drop=True)
dic = spark.table(f"{FQ}.gold_churn_dicionario").toPandas()
texto = spark.table(f"{FQ}.gold_churn_texto").toPandas()
for nome, df in [("gold_churn_treino", treino), ("gold_churn_inferencia", inferencia),
                 ("gold_churn_dicionario", dic), ("gold_churn_texto", texto)]:
    print(f"{FQ}.{nome}: {virg(len(df), 0)} linhas · {df.shape[1]} colunas")

In [ ]:
# 1.2 Teste de carga: cada regra que a modelagem supõe sobre a gold
candidatas = dic.loc[dic["papel"] == "candidata", "coluna"].tolist()
testes = [
    ("treino com 5.634 clientes", len(treino) == 5634),
    ("inferência com 1.409 clientes (D13)", len(inferencia) == 1409),
    ("mesmas colunas e ordem no treino e na inferência", list(inferencia.columns) == [c for c in treino.columns if c != "Churn"]),
    ("Churn só no treino, com 0 e 1", "Churn" not in inferencia.columns and set(treino["Churn"].unique()) == {0, 1}),
    ("dicionário cobre todas as colunas da ABT", set(dic["coluna"]) == set(treino.columns)),
    ("toda candidata tem sugestão", dic.loc[dic["papel"] == "candidata", "sugestao"].notna().all()),
    ("nenhum valor ausente", treino.isna().sum().sum() == 0 and inferencia.isna().sum().sum() == 0),
    ("customerID único nas duas bases", treino["customerID"].is_unique and inferencia["customerID"].is_unique),
    ("nenhum cliente nas duas bases", not set(treino["customerID"]) & set(inferencia["customerID"])),
    ("texto fora da ABT (D9)", "CustomerFeedback" not in treino.columns),
    ("tabela de texto cobre os dois conjuntos", len(texto) == len(treino) + len(inferencia)),
    ("taxa de churn de 26,7%", round(100 * treino["Churn"].mean(), 1) == 26.7),
]
resultado = pd.DataFrame(testes, columns=["regra", "ok"])
display(resultado)
assert resultado["ok"].all(), "a gold não está como a modelagem espera: rodar de novo o notebook 04. Falhou: " + \
    ", ".join(resultado.loc[~resultado["ok"], "regra"])
print(f"Gold conferida: {len(candidatas)} colunas candidatas, "
      f"{(dic['sugestao'] == 'recomendada').sum()} recomendadas, {(dic['sugestao'] == 'candidata').sum()} candidatas, "
      f"{(dic['sugestao'] == 'sugiro descartar').sum()} que sugiro descartar.")

> **Como ler o teste de carga**
> - **Cada linha é uma regra** que a modelagem supõe. `ok = True` quer dizer que a regra vale.
> - Se alguma der `False`, o notebook **para** e a mensagem diz qual regra falhou. A correção quase sempre é rodar de
>   novo o 04 (e, antes dele, o 02).
> - **As mais importantes:** mesmas colunas no treino e na inferência (senão o modelo não consegue prever a inferência)
>   e o texto fora da ABT (senão o modelo oficial usaria o vazamento sem querer).
> - **Resultado (11/11):** as 12 regras deram `True`; 49 colunas candidatas: 9 recomendadas, 21 candidatas e 19 que sugiro descartar.

## 2. Preparação das variáveis

> **M1 — Que variáveis entram no modelo, e por quê?** (parte 1: preparação)

### Teoria: categorias redundantes e dummies

- **Categóricas redundantes:** `OnlineSecurity` (Yes/No/No internet service) guarda a mesma informação que
  `OnlineSecurity_bin` junto com `InternetService`. Manter as duas criaria **colinearidade perfeita**: o modelo não
  conseguiria separar o efeito de uma e de outra. Saem as 6 categóricas de serviço; ficam as versões `_bin` (D10).
- **Dummies:** cada categórica vira colunas 0/1, uma por categoria **menos uma** (a referência, a primeira em ordem
  alfabética). Exemplo: `Contract` → `Contract_One year` e `Contract_Two year`; o mensal é a referência.

In [ ]:
# 2.1 Retira as 6 categóricas redundantes (51 → 45 colunas)
# 1. Variáveis categóricas redundantes
variaveis_redundantes = [
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
]

# 2. Conferir quais colunas realmente existem
remover = [col for col in variaveis_redundantes if col in treino.columns]

# 3. Criar a base sem as variáveis redundantes
df_limpo = treino.drop(columns=remover).copy()

# 4. Conferências
print("Variáveis removidas:", len(remover))
print(remover)

print("\nQuantidade de colunas antes:", treino.shape[1])
print("Quantidade de colunas depois:", df_limpo.shape[1])

print("\nColunas restantes:")
print(df_limpo.columns.tolist())

In [ ]:
# 2.2 Cardinalidade das categóricas que vão virar dummies
import pandas as pd

# 1. Identificar variáveis categóricas
variaveis_categoricas = df_limpo.select_dtypes(
    include=["object", "category", "string", "bool"]
).columns.tolist()

# 2. Retirar identificador, caso esteja entre as categóricas
variaveis_categoricas = [
    col for col in variaveis_categoricas
    if col != "customerID"
]

# 3. Criar tabela de cardinalidade
tabela_cardinalidade = pd.DataFrame({
    "Variavel": variaveis_categoricas,
    "Cardinalidade": [
        df_limpo[col].nunique(dropna=True)
        for col in variaveis_categoricas
    ],
    "Categorias": [
        df_limpo[col].dropna().unique().tolist()
        for col in variaveis_categoricas
    ],
    "Nulos": [
        df_limpo[col].isna().sum()
        for col in variaveis_categoricas
    ]
})

# 4. Ordenar pela cardinalidade
tabela_cardinalidade = tabela_cardinalidade.sort_values(
    by="Cardinalidade",
    ascending=False
).reset_index(drop=True)

display(tabela_cardinalidade)

In [ ]:
# 2.3 Cria as dummies (drop_first=True)
import pandas as pd

# 1. Identificar as variáveis categóricas
variaveis_categoricas = df_limpo.select_dtypes(
    include=["object", "category", "string", "bool"]
).columns.tolist()

# 2. Remover identificador e variável-alvo da codificação
variaveis_categoricas = [
    col for col in variaveis_categoricas
    if col not in ["customerID", "Churn"]
]

# 3. Criar variáveis dummy utilizando n-1 categorias
df_modelagem = pd.get_dummies(
    df_limpo,
    columns=variaveis_categoricas,
    drop_first=True,
    dtype=int
)

# 4. Exibir as colunas resultantes
print("Variáveis categóricas codificadas:")
print(variaveis_categoricas)

print("\nQuantidade de colunas antes:", df_limpo.shape[1])
print("Quantidade de colunas depois:", df_modelagem.shape[1])

print("\nColunas da base:")
print(df_modelagem.columns.tolist())

In [ ]:
# 2.4 Dicionário das dummies: o que cada coluna 0/1 representa e qual é a referência
# Criar dicionário das variáveis dummy
dicionario_dummies = []

for coluna in variaveis_categoricas:

    categorias = sorted(
        df_limpo[coluna].dropna().unique().tolist(),
        key=str
    )

    # Primeira categoria é a referência (drop_first=True)
    categoria_referencia = categorias[0]

    for categoria in categorias[1:]:

        nome_dummy = f"{coluna}_{categoria}"

        dicionario_dummies.append({
            "Variavel_Dummy": nome_dummy,
            "Variavel_Original": coluna,
            "Classe_Representada": categoria,
            "Interpretacao": f"1 = {categoria}; 0 = demais categorias",
            "Categoria_Referencia": categoria_referencia
        })

tabela_dicionario_dummies = pd.DataFrame(dicionario_dummies)

display(tabela_dicionario_dummies)
display(df_modelagem.head())

> **Como ler a preparação**
> - **2.1:** saem `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV` e `StreamingMovies`;
>   a base vai de 51 para 45 colunas.
> - **2.3:** as 11 categóricas restantes viram 21 dummies; a base vai para 55 colunas (53 preditoras, sem `customerID` e
>   `Churn`).
> - **2.4:** cada linha diz o que é o 1 da dummy e qual é a referência; é a tabela para ler os coeficientes depois.

### Teoria: por que separar treino e teste antes de tudo

Separo **20% dos clientes** (1.127) como **base de teste** e só uso essa base no fim, para a avaliação final. Tudo o que
é escolhido (variáveis retiradas, hiperparâmetros, limiar) usa **só o treino** (4.507). Assim o resultado no teste é
uma estimativa honesta de como o modelo vai se sair em clientes novos. A separação é **estratificada**: as duas bases
têm a mesma taxa de churn (26,7%).

In [ ]:
# 2.5 Separa treino (80%) e teste (20%), estratificado pelo Churn
from sklearn.model_selection import train_test_split
import pandas as pd

# Separar as variáveis preditoras e a variável-alvo
X = df_modelagem.drop(columns=["Churn"]).copy()
y = df_modelagem["Churn"].copy()

# Garantir que a variável-alvo seja numérica
y = pd.to_numeric(y, errors="raise").astype(int)

# Separar treino e teste
X_treino_original, X_teste_original, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Conferir dimensões
print("Treino:", X_treino_original.shape)
print("Teste:", X_teste_original.shape)

# Conferir proporção de Churn
print("\nProporção de Churn no treino:")
print(y_treino.value_counts(normalize=True))

print("\nProporção de Churn no teste:")
print(y_teste.value_counts(normalize=True))

> **Como ler a separação:** treino com 4.507 clientes (26,69% de churn) e teste com 1.127 (26,71%). A estratificação
> preservou a taxa de churn nas duas bases.

## 3. Testes estatísticos e retirada de variáveis

> **M1 — Que variáveis entram no modelo, e por quê?** (parte 2: testes e retirada)

### Teoria: a bateria de testes

Todos os testes usam **só a base de treino**:

| Teste | Para quê | Como ler |
|---|---|---|
| Correlação de Pearson entre preditoras | Achar pares que repetem informação | \|r\| acima de 0,7 = colinearidade forte |
| Pearson com o `Churn`, com p-valor | Força e direção da relação de cada variável com o churn | p-valor abaixo de 0,05 = relação significativa |
| Qui-quadrado (ou Fisher, com poucos casos) + V de Cramér | Associação das variáveis 0/1 com o churn, e o tamanho do efeito | V perto de 0 = efeito desprezível |
| Mann-Whitney | Diferença das variáveis contínuas entre quem sai e quem fica, sem supor distribuição normal | Compara as medianas dos dois grupos |
| Correção FDR (Benjamini-Hochberg) | Com 53 testes, alguns dariam "significativo" por acaso; a correção ajusta os p-valores | Usar o p-valor ajustado |
| Duplicatas exatas (`np.allclose`) | Confirmar no dado, e não só na correlação, que uma coluna é cálculo de outras | `True` = a coluna é redundante |

In [ ]:
# 3.1 Prepara as bases numéricas para os testes (Churn 0/1, só colunas numéricas e dummies)
import pandas as pd
import numpy as np

# 1. Transformar Churn em variável binária
y_treino = y_treino.copy()
y_teste = y_teste.copy()

for y in [y_treino, y_teste]:
    if y.dtype == "object" or isinstance(y.dtype, pd.StringDtype):
        y[:] = y.astype(str).str.strip().str.lower().map({
            "yes": 1,
            "no": 0,
            "1": 1,
            "0": 0,
            "true": 1,
            "false": 0
        })
    else:
        y[:] = pd.to_numeric(y, errors="coerce")

    if y.isna().any():
        raise ValueError("Existem valores de Churn que não foram convertidos para 0 ou 1.")

# 2. Selecionar variáveis numéricas e dummies
X_treino = X_treino_original.select_dtypes(include=["number", "bool"]).copy()
X_teste = X_teste_original.select_dtypes(include=["number", "bool"]).copy()

# Remover identificador e variável-alvo, caso estejam presentes
X_treino = X_treino.drop(columns=["customerID", "Churn"], errors="ignore")
X_teste = X_teste.drop(columns=["customerID", "Churn"], errors="ignore")

# Garantir que as variáveis booleanas sejam numéricas
X_treino = X_treino.astype(float)
X_teste = X_teste.astype(float)

# Garantir que treino e teste tenham as mesmas variáveis e na mesma ordem
X_teste = X_teste.reindex(columns=X_treino.columns)

print("Quantidade de variáveis analisadas:", X_treino.shape[1])
print("Quantidade de observações no treino:", X_treino.shape[0])
print("Quantidade de observações no teste:", X_teste.shape[0])

In [ ]:
# 3.2 Pares de preditoras com correlação de Pearson acima de 0,7
# Matriz de correlação
matriz_correlacao = X_treino.corr(method="pearson")

# Selecionar apenas a parte superior da matriz para evitar pares duplicados
matriz_superior = matriz_correlacao.where(
    np.triu(np.ones(matriz_correlacao.shape), k=1).astype(bool)
)

# Transformar em tabela
correlacoes_altas = (
    matriz_superior
    .stack()
    .reset_index()
)

correlacoes_altas.columns = [
    "Variavel_1",
    "Variavel_2",
    "Correlacao"
]

# Filtrar correlações com valor absoluto superior a 0.7
correlacoes_altas = correlacoes_altas[
    correlacoes_altas["Correlacao"].abs() > 0.7
].copy()

# Ordenar pela intensidade da correlação
correlacoes_altas["Correlacao_Absoluta"] = (
    correlacoes_altas["Correlacao"].abs()
)

correlacoes_altas = correlacoes_altas.sort_values(
    "Correlacao_Absoluta",
    ascending=False
).drop(columns="Correlacao_Absoluta")

display(correlacoes_altas)

In [ ]:
# 3.3 Correlação de cada preditora com o Churn, com p-valor
from scipy.stats import pearsonr

resultados_correlacao = []

for coluna in X_treino.columns:

    dados = pd.concat([X_treino[coluna], y_treino.rename("Churn")], axis=1).dropna()

    # Ignorar variáveis constantes
    if dados[coluna].nunique() < 2:
        continue

    correlacao, p_valor = pearsonr(
        dados[coluna],
        dados["Churn"]
    )

    resultados_correlacao.append({
        "Variavel": coluna,
        "Correlacao_Churn": correlacao,
        "P_valor": p_valor,
        "Significativo": "Sim" if p_valor < 0.05 else "Não",
        "Observacoes": len(dados)
    })

df_correlacao_churn = pd.DataFrame(resultados_correlacao)

# Ordenar pela intensidade da correlação
df_correlacao_churn["Correlacao_Absoluta"] = (
    df_correlacao_churn["Correlacao_Churn"].abs()
)

df_correlacao_churn = df_correlacao_churn.sort_values(
    "Correlacao_Absoluta",
    ascending=False
).drop(columns="Correlacao_Absoluta")

display(df_correlacao_churn)

In [ ]:
# 3.4 Qui-quadrado (ou Fisher), V de Cramér e Mann-Whitney, com correção FDR
from scipy.stats import chi2_contingency, fisher_exact, mannwhitneyu
from statsmodels.stats.multitest import multipletests

resultados_testes = []

for coluna in X_treino.columns:

    dados = pd.concat(
        [X_treino[coluna], y_treino.rename("Churn")],
        axis=1
    ).dropna()

    if dados[coluna].nunique() < 2:
        continue

    # Separar variáveis binárias das contínuas
    if dados[coluna].nunique() == 2:

        tabela = pd.crosstab(
            dados[coluna],
            dados["Churn"]
        ).reindex(index=[0, 1], columns=[0, 1], fill_value=0)

        chi2, p_chi2, _, esperadas = chi2_contingency(
            tabela,
            correction=False
        )

        # Fisher quando houver frequência esperada pequena
        if (esperadas < 5).any():
            estatistica, p_valor = fisher_exact(tabela)
            teste = "Fisher"
        else:
            estatistica = chi2
            p_valor = p_chi2
            teste = "Qui-quadrado"

        # Taxa de Churn entre os grupos 0 e 1
        taxa_churn_0 = dados.loc[
            dados[coluna] == 0, "Churn"
        ].mean()

        taxa_churn_1 = dados.loc[
            dados[coluna] == 1, "Churn"
        ].mean()

        diferenca = taxa_churn_1 - taxa_churn_0

        # Tamanho de efeito: V de Cramér
        n = tabela.to_numpy().sum()
        denominador = n * min(tabela.shape[0] - 1, tabela.shape[1] - 1)

        cramers_v = np.sqrt(chi2 / denominador) if denominador > 0 else np.nan

        resultados_testes.append({
            "Variavel": coluna,
            "Tipo_Teste": teste,
            "P_valor": p_valor,
            "Efeito": cramers_v,
            "Taxa_Churn_0": taxa_churn_0,
            "Taxa_Churn_1": taxa_churn_1,
            "Diferenca_Churn": diferenca,
            "Observacoes": len(dados)
        })

    else:

        grupo_0 = dados.loc[
            dados["Churn"] == 0, coluna
        ]

        grupo_1 = dados.loc[
            dados["Churn"] == 1, coluna
        ]

        estatistica, p_valor = mannwhitneyu(
            grupo_0,
            grupo_1,
            alternative="two-sided"
        )

        resultados_testes.append({
            "Variavel": coluna,
            "Tipo_Teste": "Mann-Whitney",
            "P_valor": p_valor,
            "Mediana_Churn_0": grupo_0.median(),
            "Mediana_Churn_1": grupo_1.median(),
            "Observacoes": len(dados)
        })


df_testes = pd.DataFrame(resultados_testes)

# Correção para múltiplos testes
df_testes["P_valor_ajustado"] = multipletests(
    df_testes["P_valor"],
    method="fdr_bh"
)[1]

# Classificação da significância
df_testes["Significativo"] = (
    df_testes["P_valor_ajustado"] < 0.05
)

df_testes = df_testes.sort_values(
    "P_valor_ajustado"
)

display(df_testes)

In [ ]:
# 3.5 Qui-quadrado das variáveis 0/1, com correção FDR (conferência)
from scipy.stats import chi2_contingency

resultados_categoricas = []

for coluna in X_treino.columns:

    dados = pd.DataFrame({
        coluna: X_treino[coluna],
        "Churn": y_treino
    }).dropna()

    # Considerar apenas variáveis binárias (dummies)
    if dados[coluna].nunique() != 2:
        continue

    tabela = pd.crosstab(
        dados[coluna],
        dados["Churn"]
    ).reindex(index=[0, 1], columns=[0, 1], fill_value=0)

    chi2, p_valor, graus_liberdade, esperadas = chi2_contingency(
        tabela
    )

    n = tabela.to_numpy().sum()

    cramers_v = np.sqrt(
        chi2 / (n * min(tabela.shape[0] - 1, tabela.shape[1] - 1))
    )

    resultados_categoricas.append({
        "Variavel": coluna,
        "P_valor": p_valor,
        "Graus_Liberdade": graus_liberdade,
        "Cramers_V": cramers_v,
        "Observacoes": n
    })

df_categoricas = pd.DataFrame(resultados_categoricas)

# Correção para múltiplos testes
df_categoricas["P_valor_ajustado"] = multipletests(
    df_categoricas["P_valor"],
    method="fdr_bh"
)[1]

df_categoricas["Significativo"] = (
    df_categoricas["P_valor_ajustado"] < 0.05
)

df_categoricas = df_categoricas.sort_values(
    "P_valor_ajustado"
)

display(df_categoricas)

> **Como ler os testes**
> - **3.2:** os pares acima de 0,7 apontam as candidatas a redundância (por exemplo, as várias versões do tempo de casa).
> - **3.3:** as mais ligadas ao churn são `is_monthly_contract` (cerca de +0,40), `IsHighRiskContract` (+0,36),
>   `HasFiberOpticWithoutTechSupport` (+0,35), `tenure` (−0,34) e `no_support_with_internet` (+0,34).
> - **3.4 e 3.5:** depois da correção FDR, `gender_Male` não é significativa (p-ajustado bem acima de 0,05 e V de Cramér
>   perto de zero). Os números exatos dependem de quem caiu no treino: a justificativa escrita na tabela da 3.11
>   (p-ajustado 0,152; V 0,022) é a da execução original, e com o treino ordenado por `customerID` (1.1) os valores da
>   3.4 mudam um pouco, sem mudar a conclusão. Dummies
>   não significativas **dentro** de blocos que ficam (por exemplo, `TenureCohort_13-24`) são mantidas, para não quebrar
>   a codificação da categoria de referência.

In [ ]:
# 3.6 Estatísticas das variáveis candidatas a redundância
# Comparar as variáveis mais correlacionadas
colunas_verificacao = [
    "MonthlyIncome",
    "income_after_charges",
    "MonthlyCharges",
    "average_monthly_charges",
    "TotalServicesSubscribed",
    "total_addons",
    "n_protecao",
    "n_streaming",
    "is_new_customer",
    "new_monthly_customer",
    "has_family",
    "Partner_Yes",
    "StreamingTV_bin",
    "StreamingMovies_bin",
    "ChargesRatio",
    "charge_difference"
]

colunas_existentes = [
    col for col in colunas_verificacao
    if col in X_treino.columns
]

display(X_treino[colunas_existentes].describe().T)

X_treino[[
    "MonthlyIncome",
    "income_after_charges",
    "MonthlyCharges",
    "average_monthly_charges",
    "TotalServicesSubscribed",
    "total_addons",
    "n_protecao",
    "n_streaming",
    "is_new_customer",
    "new_monthly_customer",
    "has_family",
    "ChargesRatio",
    "charge_difference"
]].describe().T

In [ ]:
# 3.7 Recalcula as derivadas para comparar com as gravadas
comparacoes = pd.DataFrame({
    "income_menos_charges": (
        X_treino["MonthlyIncome"] -
        X_treino["MonthlyCharges"]
    ),

    "soma_addons": (
        X_treino["n_protecao"] +
        X_treino["n_streaming"]
    ),

    "media_mensal_calculada": (
        X_treino["TotalCharges"] /
        X_treino["tenure"].replace(0, np.nan)
    )
})

comparacoes["income_after_charges"] = X_treino["income_after_charges"]
comparacoes["total_addons"] = X_treino["total_addons"]
comparacoes["average_monthly_charges"] = X_treino["average_monthly_charges"]

comparacoes.describe().T

In [ ]:
# 3.8 Duplicatas exatas: income_after_charges e total_addons
print(
    "income_after_charges é igual à renda menos mensalidade:",
    np.allclose(
        X_treino["income_after_charges"],
        X_treino["MonthlyIncome"] - X_treino["MonthlyCharges"],
        equal_nan=True
    )
)

print(
    "total_addons é igual à soma das contagens:",
    np.allclose(
        X_treino["total_addons"],
        X_treino["n_protecao"] + X_treino["n_streaming"],
        equal_nan=True
    )
)

In [ ]:
# 3.9 average_monthly_charges = TotalCharges ÷ tenure? (diferenças só onde tenure = 0)
comparacao = pd.DataFrame({
    "original": X_treino["average_monthly_charges"],
    "calculada": (
        X_treino["TotalCharges"] /
        X_treino["tenure"].replace(0, np.nan)
    )
})

comparacao["diferenca"] = (
    comparacao["original"] - comparacao["calculada"]
)

comparacao["diferenca_abs"] = comparacao["diferenca"].abs()

display(comparacao.describe().T)

print(
    "Registros iguais:",
    np.isclose(
        comparacao["original"],
        comparacao["calculada"],
        equal_nan=True
    ).sum()
)

print(
    "Registros diferentes:",
    (~np.isclose(
        comparacao["original"],
        comparacao["calculada"],
        equal_nan=True
    )).sum()
)

In [ ]:
# 3.10 Os clientes com tenure = 0
display(
    X_treino.loc[
        X_treino["tenure"] == 0,
        [
            "tenure",
            "TotalCharges",
            "MonthlyCharges",
            "average_monthly_charges"
        ]
    ]
)

> **Como ler as duplicatas**
> - `income_after_charges` = `MonthlyIncome` − `MonthlyCharges` e `total_addons` = `n_protecao` + `n_streaming` em
>   **100%** das linhas: são cálculos exatos de outras colunas.
> - `average_monthly_charges` = `TotalCharges` ÷ `tenure` em 4.498 clientes; os 9 diferentes são justamente os de
>   `tenure` = 0, em que ela assume a mensalidade.

In [ ]:
# 3.11 As 12 variáveis retiradas, com a justificativa de cada uma
df_variaveis_retiradas = pd.DataFrame([
    {
        "Variavel": "tenure_years",
        "Justificativa": "Representa tenure convertido de meses para anos. Como é uma transformação direta, manteremos tenure."
    },
    {
        "Variavel": "is_electronic_check",
        "Justificativa": "É uma duplicata exata de PaymentMethod_Electronic check. Manteremos a variável original."
    },
    {
        "Variavel": "PhoneService_Yes",
        "Justificativa": "É o inverso de MultipleLines_No phone service. As duas variáveis representam a mesma informação."
    },
    {
        "Variavel": "tenure_group_13-24",
        "Justificativa": "É uma duplicata exata de TenureCohort_13-24. Manteremos uma das representações."
    },
    {
        "Variavel": "income_after_charges",
        "Justificativa": "É calculada exatamente como MonthlyIncome - MonthlyCharges. Manteremos as duas variáveis originais."
    },
    {
        "Variavel": "total_addons",
        "Justificativa": "É calculada exatamente como n_protecao + n_streaming. Manteremos as variáveis componentes."
    },
    {
        "Variavel": "average_monthly_charges",
        "Justificativa": "É calculada por TotalCharges / tenure quando tenure é maior que zero. Nos registros com tenure igual a zero, assume o mesmo valor de MonthlyCharges. Portanto, sua informação já está representada nas demais variáveis."
    },
    {
        "Variavel": "tenure_group_7-12",
        "Justificativa": "Segunda discretização de tenure, com correlação de 0.923 com TenureCohort_6-12. A base tinha três versões do mesmo sinal (tenure numérico + TenureCohort + tenure_group). Manteremos tenure e apenas um binning (TenureCohort). Evidência do problema (na execução original): no Lasso, TenureCohort_6-12 (-0.222) e tenure_group_7-12 (+0.174) receberam sinais opostos — artefato típico de colinearidade que distorce a interpretação dos coeficientes."
    },
    {
        "Variavel": "tenure_group_25-48",
        "Justificativa": "Segunda discretização de tenure. A codificação tripla do mesmo sinal dilui a importância da variável nos modelos de árvore e instabiliza os coeficientes nos modelos lineares. A informação já está coberta por tenure e TenureCohort."
    },
    {
        "Variavel": "tenure_group_49+",
        "Justificativa": "Segunda discretização de tenure, com correlação de 0.613 com TenureCohort_25+. Redundante pelo mesmo motivo das demais variáveis tenure_group."
    },
    {
        "Variavel": "is_monthly_contract",
        "Justificativa": "Combinação linear exata dos dummies de Contract: como Month-to-month é a categoria de referência, is_monthly_contract = 1 - Contract_One year - Contract_Two year (verificado na base de treino, com diferença máxima de 0.0). É o mesmo tipo de duplicata exata já removida nos outros casos; a informação está integralmente nos dummies de Contract."
    },
    {
        "Variavel": "gender_Male",
        "Justificativa": "Não significativa após correção para múltiplos testes (FDR/Benjamini-Hochberg): p-valor ajustado de 0.467, acima do limiar de 0.05, com efeito desprezível (V de Cramer de 0.011). Não há evidência estatística de associação com Churn na base de treino."
    }
])

display(df_variaveis_retiradas)

In [ ]:
# 3.12 Bases finais: 41 variáveis, as mesmas no treino e no teste
# Lista de variáveis que serão retiradas
variaveis_retirar = df_variaveis_retiradas["Variavel"].tolist()

# Criar novas bases
X_treino_limpo = X_treino.drop(
    columns=variaveis_retirar,
    errors="ignore"
).copy()

X_teste_limpo = X_teste.drop(
    columns=variaveis_retirar,
    errors="ignore"
).copy()

# Conferir dimensões
print("Base de treino:", X_treino_limpo.shape)
print("Base de teste:", X_teste_limpo.shape)

# Conferir se as colunas são iguais
print(
    "Mesmas variáveis nas duas bases:",
    X_treino_limpo.columns.equals(X_teste_limpo.columns)
)

> **Como ler a retirada e resposta (M1)**
> - **Saem 12 variáveis:** 7 por serem cálculo exato ou inverso de outras (`is_electronic_check`, `PhoneService_Yes`,
>   `tenure_group_13-24`, `income_after_charges`, `total_addons`, `average_monthly_charges`, `is_monthly_contract`), 4 por
>   serem outra versão do tempo de casa (`tenure_years` e três `tenure_group`), e `gender_Male` por não ter associação
>   com o churn depois da correção FDR.
> - **Resposta:** entram **41 variáveis**, em seis temas (fidelidade, contrato, serviços, pagamento, financeiro e
>   perfil), sem colunas que repetem outras exatamente.

In [ ]:
# 3.13 Grava a separação treino/teste e as listas de variáveis, para o 06 reconstruir exatamente as mesmas bases
# a ordem das linhas dentro do treino importa: a validação cruzada do 06 divide as dobras por posição
separacao = pd.concat([
    pd.DataFrame({"customerID": X_treino_original["customerID"].to_numpy(), "base": "Treino",
                  "ordem": np.arange(len(X_treino_original))}),
    pd.DataFrame({"customerID": X_teste_original["customerID"].to_numpy(), "base": "Teste",
                  "ordem": np.arange(len(X_teste_original))}),
], ignore_index=True)
listas = pd.concat([
    pd.DataFrame({"tipo": "redundante", "variavel": variaveis_redundantes,
                  "justificativa": "Categórica de serviço com 3 níveis; fica a versão sim/não (_bin), D10"}),
    pd.DataFrame({"tipo": "categorica_dummy", "variavel": variaveis_categoricas,
                  "justificativa": "Vira colunas 0/1 (dummies), sem a primeira categoria (referência)"}),
    df_variaveis_retiradas.rename(columns={"Variavel": "variavel", "Justificativa": "justificativa"}).assign(tipo="retirada"),
    pd.DataFrame({"tipo": "modelo", "variavel": X_treino_limpo.columns.tolist(), "justificativa": "Entra no modelo"}),
], ignore_index=True)[["tipo", "variavel", "justificativa"]]
listas.insert(1, "ordem", listas.groupby("tipo").cumcount() + 1)

for nome, df, comentario in [
    ("gold_churn_split", separacao, "Separação treino (80%) e teste (20%) do notebook 05, estratificada pelo Churn, com a ordem das linhas"),
    ("gold_churn_variaveis_modelo", listas, "Variáveis do modelo (notebook 05): redundantes, categóricas que viram dummies, retiradas e as 41 do modelo"),
]:
    spark.createDataFrame(df).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{FQ}.{nome}")
    spark.sql(f"COMMENT ON TABLE {FQ}.{nome} IS '{comentario}'")
    print(f"{FQ}.{nome}: {virg(len(df), 0)} linhas")
display(listas.groupby("tipo", sort=False).size().rename("variaveis").reset_index())

> **Como ler a gravação:** `gold_churn_split` tem uma linha por cliente do treino da gold (4.507 de treino e
> 1.127 de teste) e `gold_churn_variaveis_modelo` tem as quatro listas. O 06 lê as duas e reconstrói as bases sem
> repetir os testes.

## Conferência com os métodos da aula (seções 4 e 5)

As seções 2 e 3 chegaram às **41 colunas do modelo** por testes estatísticos, uma variável (ou um par) de cada vez.
Aqui faço a pergunta pelo outro lado: **num modelo com todas juntas, quais variáveis acrescentam informação?** Uso os
métodos da disciplina de regressão (seleção para frente por AUC, *stepwise* por AIC, BIC e p-valor, VIF e intervalo
dos coeficientes), com uma regressão logística simples e a validação cruzada repetida (5 dobras × 5 repetições).

**Por que conferir:** se dois caminhos diferentes chegam às mesmas variáveis, a escolha não depende do método. A seção 6
mostra o encontro. As variáveis aqui saem da lista da triagem do 04 (e não das 41), porque o modelo simples precisa de
uma versão de cada informação; por isso a conferência usa, por exemplo, `is_electronic_check`, que nas 41 aparece como
`PaymentMethod_Electronic check` (a mesma coluna, 3.11).

**Base usada:** os 5.634 clientes do histórico, com validação cruzada. A conferência não alimenta o modelo (que só
vê o treino da 2.5); ela serve para checar a escolha das seções 2 e 3.

## 4. Preparar as variáveis para o modelo

### Teoria: transformar categorias em números (*dummies*)

A regressão logística só trabalha com números. Cada variável de categorias vira **colunas 0/1, uma por categoria,
menos uma** (a categoria de referência). Exemplo com `Contract`:

| Contract | `Contract_One year` | `Contract_Two year` |
|---|---|---|
| Month-to-month (referência) | 0 | 0 |
| One year | 1 | 0 |
| Two year | 0 | 1 |

- **Por que menos uma:** se as três colunas entrassem, a soma delas seria sempre 1, igual à constante do modelo
  (o intercepto), e o modelo não conseguiria separar o efeito de cada uma (a *armadilha da variável dummy*). A
  categoria que fica de fora vira a **referência**: os coeficientes das outras dizem o quanto o risco muda **em
  relação a ela**.
- **Qual é a referência:** a primeira categoria em ordem alfabética. Nas variáveis da seleção: `Contract` →
  Month-to-month; `InternetService` → DSL; `Partner`, `Dependents` e `PaperlessBilling` → No. Assim, "fibra" no
  modelo quer dizer "fibra em comparação com DSL".
- **As categorias vêm do treino:** a inferência usa exatamente as mesmas colunas, mesmo que alguma categoria não
  apareça nela. É a regra "tudo o que é aprendido vem do treino" (seção 2 do 04).
- **Escala dos números:** antes da regressão, os números são padronizados (média 0, desvio 1), para que `tenure`
  (0 a 72) e `mensalidade_relativa_pacote` (perto de 1) pesem na mesma escala.

In [ ]:
# 4.1 Função que monta a matriz do modelo (as categorias e a ordem das colunas vêm sempre do treino)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score

y = treino["Churn"]
eh_texto = lambda c: not pd.api.types.is_numeric_dtype(treino[c])
CATEGORIAS = {c: sorted(treino[c].unique()) for c in candidatas if eh_texto(c)}

def matriz(df: pd.DataFrame, colunas: list) -> pd.DataFrame:
    X = df[colunas].copy()
    cat = [c for c in colunas if c in CATEGORIAS]
    for c in cat:
        X[c] = pd.Categorical(X[c], categories=CATEGORIAS[c])
    return pd.get_dummies(X, columns=cat, drop_first=True, dtype=float)

exemplo = matriz(treino, ["Contract", "tenure"])
print("Exemplo: Contract e tenure viram", list(exemplo.columns))
assert list(matriz(inferencia, ["Contract", "tenure"]).columns) == list(exemplo.columns), "colunas diferentes na inferência"

## 5. Seleção pelos métodos da aula (conferência)

### Teoria: como medir se um modelo é bom (AUC)

O modelo dá, para cada cliente, uma **nota de risco** (a probabilidade de sair). A **AUC** (*Area Under the Curve*,
área sob a curva ROC) responde a uma pergunta simples:

> **Sorteando um cliente que saiu e um que ficou, qual a chance de o modelo dar nota maior a quem saiu?**

| AUC | Leitura |
|---|---|
| 0,50 | Sorteio: o modelo não separa nada |
| 0,70 | Separa razoavelmente |
| 0,80 | Separa bem |
| 0,90 ou mais | Separa muito bem; em churn, desconfiar de vazamento |

A AUC não depende de um ponto de corte: avalia a **ordem** das notas. O ponto de corte (a partir de que nota acionar
o cliente) é decidido depois, pelo custo da ação (notebook 00 e modelagem).

### Teoria: validação cruzada (por que não medir no próprio treino)

Medir o acerto nos mesmos clientes usados para treinar é como corrigir a prova com o gabarito na mão: o resultado sai
otimista. Na **validação cruzada com 5 dobras**, divido os 5.634 clientes em 5 partes, treino com 4 e meço na quinta,
repetindo 5 vezes. A AUC final é a média das 5 medidas, sempre em clientes que o modelo **não viu**. As dobras são
**estratificadas**: cada uma tem os mesmos 26,7% de churn da base.

**Validação cruzada repetida:** quais clientes caem em cada dobra é sorte do sorteio. Com ganhos pequenos, essa sorte
pode decidir se uma variável entra ou não (aconteceu: a mesma variável ganhou 0,0025 numa ordem das linhas e 0,0020 em
outra). Por isso repito o sorteio 5 vezes: **5 dobras × 5 repetições = 25 medidas**. A AUC é a média das 25, e para
cada candidata também conto **em quantas das 25 dobras ela melhorou o modelo** (a consistência do ganho).

### Teoria: seleção para frente (*forward*) por grupos de alternativas

1. **Começo com as 9 recomendadas** do dicionário (seção 3 do 04).
2. **Testo cada candidata**, uma de cada vez, junto com as que já entraram, e meço a AUC na validação cruzada.
3. **A que mais melhora entra.** Repito o passo 2 com as que sobraram.
4. **Paro quando nenhuma melhora a AUC em pelo menos 0,002.** É um critério prático: abaixo disso, o ganho é pequeno
   demais para justificar uma variável a mais, que deixa o modelo mais difícil de explicar.

5. **Na fronteira (ganho entre 0,001 e 0,002), decido com estatística e negócio juntos** (5.2b): a candidata entra
   se o ganho for consistente (melhora em pelo menos 80% das 25 dobras), se o coeficiente dela tiver intervalo de 95%
   sem o zero, se não repetir as outras (VIF abaixo de 5) e se tiver uma leitura de negócio clara.

**Um cuidado:** as mesmas dobras servem para **escolher** as variáveis e para **medir** a AUC da seleção. Por isso a
AUC final sai levemente otimista: entre muitas candidatas testadas, a que ganhou pode ter tido um pouco de sorte nas
dobras. Com ganhos tão pequenos, o efeito é pequeno, mas é o motivo de a modelagem comparar o modelo final de novo com
outros, e de a seleção ser uma sugestão.

**Grupos de alternativas:** variáveis que dizem a mesma coisa (por exemplo, `PaymentMethod`, `is_electronic_check` e
`pagamento_automatico`) formam um grupo. **Quando uma entra, as outras do grupo saem da disputa**: colocar duas
repetiria informação (multicolinearidade). As que eu sugeri descartar no 04 não entram na disputa.

In [ ]:
# 5.1 Ponto de partida (as recomendadas) e grupos de alternativas entre as candidatas
RECOMENDADAS = dic.loc[dic["sugestao"] == "recomendada", "coluna"].tolist()
GRUPOS = {                                           # variáveis que dizem a mesma coisa: só uma de cada grupo entra
    "forma de pagamento": ["PaymentMethod", "is_electronic_check", "pagamento_automatico"],
    "preço": ["MonthlyCharges", "mensalidade_relativa_pacote"],
    "mudança de preço": ["charge_difference", "ChargesRatio"],
    "backup": ["OnlineBackup_bin"],
    "proteção de aparelho": ["DeviceProtection_bin"],
    "nº de serviços de proteção": ["n_protecao"],
    "contrato mensal e cheque eletrônico": ["IsHighRiskContract"],
    "novo e mensal": ["new_monthly_customer"],
    "cliente novo": ["is_new_customer"],
    "internet sem suporte": ["no_support_with_internet"],
    "fibra sem suporte": ["HasFiberOpticWithoutTechSupport"],
    "TotalCharges ausente": ["fl_total_charges_ausente"],
}
# conferências: tudo o que está nos grupos é candidata; nenhuma recomendada está nos grupos
em_grupo = [v for g in GRUPOS.values() for v in g]
assert set(em_grupo) <= set(dic.loc[dic["sugestao"] == "candidata", "coluna"]), "grupo com variável que não é candidata"
assert not set(em_grupo) & set(RECOMENDADAS)
fora = sorted(set(dic.loc[dic["sugestao"] == "candidata", "coluna"]) - set(em_grupo))
print("Recomendadas (ponto de partida):", ", ".join(RECOMENDADAS))
print("Candidatas fora da disputa, porque a versão recomendada já está no ponto de partida:", ", ".join(fora))

# matriz de todas as candidatas montada uma vez só; cada modelo pega as colunas de que precisa (mais rápido)
TODAS = dic.loc[dic["sugestao"].isin(["recomendada", "candidata"]), "coluna"].tolist()   # as 30 que não sugerimos descartar
X_TODAS = matriz(treino, TODAS)
COLUNAS_DE = {v: [c for c in X_TODAS.columns if c == v or c.startswith(f"{v}_")] for v in TODAS}
for v in TODAS:                                       # "OnlineSecurity_" pegaria também "OnlineSecurity_bin": separo
    COLUNAS_DE[v] = [c for c in COLUNAS_DE[v] if c == v or c[len(v) + 1:] in [str(k) for k in CATEGORIAS.get(v, [])]]
def mat(colunas: list) -> pd.DataFrame:
    """As colunas do modelo (com as dummies) para uma lista de variáveis do treino."""
    return X_TODAS[[c for v in colunas for c in COLUNAS_DE[v]]]

# validação cruzada repetida: 5 dobras × 5 repetições = 25 medidas, sempre as mesmas para todos os modelos
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.linear_model import LinearRegression
CV = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)
_notas = {}
def notas_cv(colunas: list) -> np.ndarray:
    """AUC em cada uma das 25 dobras (guardada, para não recalcular)."""
    chave = tuple(colunas)
    if chave not in _notas:
        modelo = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
        _notas[chave] = cross_val_score(modelo, mat(colunas).values, y, cv=CV, scoring="roc_auc")
    return _notas[chave]
def auc_cv(colunas: list) -> tuple:
    n = notas_cv(colunas)
    return n.mean(), n.std()

def ajustar_logistica(X: pd.DataFrame) -> tuple:
    """Logística sem penalização (como o glm do R), pelo método de Newton (IRLS), com as colunas padronizadas.
    Devolve coeficientes (com intercepto), probabilidades, matriz de covariância e o posto da matriz."""
    Z = ((X - X.mean()) / X.std(ddof=0).replace(0, 1)).values
    A = np.column_stack([np.ones(len(Z)), Z])
    b = np.zeros(A.shape[1])
    yv = np.asarray(y, dtype=float)
    for _ in range(50):
        p = 1 / (1 + np.exp(-(A @ b)))
        H = A.T @ (A * (p * (1 - p))[:, None])
        passo = np.linalg.lstsq(H, A.T @ (yv - p), rcond=None)[0]
        b += passo
        if np.abs(passo).max() < 1e-8:
            break
    p = np.clip(1 / (1 + np.exp(-(A @ b))), 1e-12, 1 - 1e-12)
    cov = np.linalg.pinv(A.T @ (A * (p * (1 - p))[:, None]))
    return b, p, cov, np.linalg.matrix_rank(A)

def coeficientes(colunas: list) -> pd.DataFrame:
    """Coeficientes padronizados (colunas com média 0 e desvio 1) e intervalo de 95% pelo erro-padrão da logística."""
    X = mat(colunas)
    b, _, cov, _ = ajustar_logistica(X)
    ep = np.sqrt(np.diag(cov))[1:]
    return pd.DataFrame({"coluna": X.columns, "coef": b[1:], "inf": b[1:] - 1.96 * ep, "sup": b[1:] + 1.96 * ep}).set_index("coluna")

def vif_tabela(colunas: list) -> pd.DataFrame:
    """Para cada coluna do modelo: R² contra as outras colunas, VIF = 1 ÷ (1 − R²) e tolerância = 1 − R² (seção 5.4)."""
    X = mat(colunas)
    linhas = []
    for col in X.columns:
        outras = X.drop(columns=col)
        r2 = LinearRegression().fit(outras, X[col]).score(outras, X[col])
        linhas.append({"coluna_do_modelo": col, "R2_com_as_outras": r2,
                       "VIF": 1 / (1 - r2) if r2 < 1 else np.inf, "tolerancia": 1 - r2})
    t = pd.DataFrame(linhas).sort_values("VIF", ascending=False)
    t["leitura"] = pd.cut(t["VIF"], [0, 5, 10, np.inf], labels=["aceitável", "atenção", "repetição demais"])
    return t

> **Por que algumas candidatas ficam fora da disputa:** a versão recomendada delas já está no ponto de partida.
> `Contract` está nas recomendadas, então `is_monthly_contract` (que é o contrato resumido em 1/0) não acrescenta. O
> mesmo vale para os serviços originais de 3 categorias: as versões `_bin` de segurança e suporte já estão, junto com
> `InternetService` (D10).

In [ ]:
# 5.2 Seleção para frente: a cada passo, entra a candidata que mais melhora a AUC na validação cruzada repetida
GANHO_MINIMO = 0.002
selecionadas = list(RECOMENDADAS)                    # ponto de partida: as 9 recomendadas
assert len(selecionadas) == 9, "o ponto de partida precisa ser as 9 recomendadas: rodar de novo a célula 5.1"

def testar_candidatas(base: list, grupos: dict) -> pd.DataFrame:
    """Para cada candidata dos grupos livres: ganho médio de AUC e em quantas das 25 dobras ela melhorou o modelo."""
    n_base = notas_cv(base)
    linhas = []
    total = sum(len(m) for m in grupos.values())
    for grupo, membros in grupos.items():
        for v in membros:
            n_v = notas_cv(base + [v])
            print(f"\r  testando candidata {len(linhas) + 1} de {total}: {v:<35}", end="", flush=True)   # progresso
            linhas.append({"candidata": v, "grupo": grupo, "AUC": n_v.mean(), "ganho": (n_v - n_base).mean(),
                           "pct_dobras_que_melhora": 100 * ((n_v - n_base) > 0).mean()})
    print()
    return pd.DataFrame(linhas).sort_values("ganho", ascending=False).reset_index(drop=True)

auc_atual, dp_atual = auc_cv(selecionadas)
passos = [{"passo": 0, "entrou": "as 9 recomendadas", "grupo": "ponto de partida", "AUC": auc_atual,
           "desvio_entre_dobras": dp_atual, "ganho": np.nan, "pct_dobras_que_melhora": np.nan}]
import time
inicio = time.time()
grupos_livres = dict(GRUPOS)
while grupos_livres:
    print(f"Passo {len(passos)} ({virg(time.time() - inicio, 0)} s até aqui)")
    teste = testar_candidatas(selecionadas, grupos_livres)
    melhor = teste.iloc[0]
    if melhor["ganho"] < GANHO_MINIMO:
        break
    selecionadas.append(melhor["candidata"])
    grupos_livres.pop(melhor["grupo"])
    passos.append({"passo": len(passos), "entrou": melhor["candidata"], "grupo": melhor["grupo"], "AUC": melhor["AUC"],
                   "desvio_entre_dobras": auc_cv(selecionadas)[1], "ganho": melhor["ganho"],
                   "pct_dobras_que_melhora": melhor["pct_dobras_que_melhora"]})

passos = pd.DataFrame(passos)
display(passos.round(4))
print("Melhores que ficaram de fora (ganho abaixo de 0,002):")
display(teste.head(5).round(4))
print(f"Seleção pelo critério da AUC ({len(selecionadas)} variáveis): {', '.join(selecionadas)}")

> **Como ler a tabela da seleção**
> - **Passo 0** é o modelo só com as 9 recomendadas. Cada passo seguinte acrescenta a variável que mais melhorou a AUC.
> - **`AUC`** é a média das 25 dobras; **`desvio_entre_dobras`** mostra o quanto ela varia de uma dobra para outra
>   (cerca de 0,01: algumas partes da base são mais fáceis de prever que outras).
> - **Por que um ganho de 0,002 conta, se o desvio é 0,01:** todos os modelos são medidos **nas mesmas 25 dobras**.
>   A comparação é sempre na mesma dobra (pareada), então a variação entre dobras afeta os dois modelos igual e se
>   cancela. A coluna **`pct_dobras_que_melhora`** mostra isso: em quantas das 25 dobras o modelo com a candidata foi
>   melhor que sem ela.
> - **`ganho`** é quanto a AUC subiu naquele passo. Pelo critério da AUC, a seleção para quando o melhor ganho fica
>   abaixo de 0,002.
> - **As que ficaram de fora** com ganho entre 0,001 e 0,002 estão na fronteira: a 5.2b decide sobre elas.

### 5.2b Decisão na fronteira: estatística e negócio juntos

O critério da AUC é um ponto de corte prático, e uma variável com ganho de 0,0019 não é muito diferente de uma com
0,0021. Para as candidatas da **fronteira (ganho entre 0,001 e 0,002)**, a decisão junta quatro evidências:

| Critério | Pergunta | Regra |
|---|---|---|
| Consistência | O ganho aparece na maioria das dobras, ou foi sorte de algumas? | Melhora em pelo menos 80% das 25 dobras |
| Significância e tamanho do efeito | O coeficiente é diferente de zero, com folga? | Intervalo de 95% sem o zero (5.5c explica o intervalo) |
| Repetição | Ela repete variáveis que já estão no modelo? | VIF máximo abaixo de 5 (5.4) |
| Negócio | Ela vira uma ação de retenção clara? | Leitura de negócio escrita (abaixo) |

É o que a aula de regressão chama de **significância × tamanho do efeito**: o teste diz se há efeito; o ganho e o
coeficiente dizem se ele importa. Uma candidata por vez: a que passa nos quatro critérios com o maior ganho entra, e
as outras são testadas de novo, já com ela no modelo.

In [ ]:
# 5.2b Decisão na fronteira: as candidatas com ganho entre 0,001 e 0,002 passam por quatro critérios
LEITURA_NEGOCIO = {                                  # o que a variável vira em ação, se entrar no modelo
    "is_electronic_check": "oferecer desconto para trocar o cheque eletrônico por débito automático ou cartão",
    "PaymentMethod": "a mesma ação do cheque eletrônico, com 3 colunas em vez de 1",
    "pagamento_automatico": "a mesma ação, olhando pelo lado de quem já paga no automático",
    "IsHighRiskContract": "regra pronta (mensal e cheque eletrônico), mas repete contrato e pagamento",
    "MonthlyCharges": "preço: repete o tipo de internet (fibra é mais cara), sem ação nova",
    "mensalidade_relativa_pacote": "pagar acima da média do próprio pacote; repete o preço",
    "fl_total_charges_ausente": "falha de cadastro; sem ação de retenção",
}

FAIXA_FRONTEIRA = (0.001, GANHO_MINIMO)
avaliacoes = []
while grupos_livres:
    teste = testar_candidatas(selecionadas, grupos_livres)
    fronteira = teste[teste["ganho"].between(*FAIXA_FRONTEIRA)].copy()
    if fronteira.empty:
        break
    linhas = []
    for _, r in fronteira.iterrows():
        v = r["candidata"]
        coef_v = coeficientes(selecionadas + [v]).filter(regex=f"^{v}", axis=0)
        ic_sem_zero = bool(((coef_v["inf"] > 0) | (coef_v["sup"] < 0)).all())
        vif_max = vif_tabela(selecionadas + [v])["VIF"].max()
        criterios = {"consistente": r["pct_dobras_que_melhora"] >= 80, "IC sem zero": ic_sem_zero,
                     "VIF < 5": vif_max < 5, "leitura de negócio": v in LEITURA_NEGOCIO and "sem ação" not in LEITURA_NEGOCIO[v]
                     and "repete" not in LEITURA_NEGOCIO[v]}
        linhas.append({"rodada": len(avaliacoes) + 1, "candidata": v, "ganho": r["ganho"],
                       "pct_dobras_que_melhora": r["pct_dobras_que_melhora"], "VIF_max": vif_max, **criterios,
                       "leitura_de_negocio": LEITURA_NEGOCIO.get(v, "—"), "entra": all(criterios.values())})
    rodada = pd.DataFrame(linhas)
    avaliacoes.append(rodada)
    aprovadas = rodada[rodada["entra"]]
    if aprovadas.empty:
        break
    v = aprovadas.iloc[0]["candidata"]
    grupo = GRUPO_DE_CANDIDATA = next(g for g, m in grupos_livres.items() if v in m)
    selecionadas.append(v)
    grupos_livres.pop(grupo)
    passos = pd.concat([passos, pd.DataFrame([{"passo": len(passos), "entrou": v, "grupo": f"{grupo} (fronteira, 5.2b)",
                        "AUC": auc_cv(selecionadas)[0], "desvio_entre_dobras": auc_cv(selecionadas)[1],
                        "ganho": aprovadas.iloc[0]["ganho"], "pct_dobras_que_melhora": aprovadas.iloc[0]["pct_dobras_que_melhora"]}])],
                       ignore_index=True)

display(pd.concat(avaliacoes, ignore_index=True).round(4) if avaliacoes else "nenhuma candidata na fronteira")
display(passos.round(4))
print(f"Seleção da conferência ({len(selecionadas)} variáveis): {', '.join(selecionadas)}")

> **Como ler a decisão na fronteira**
> - **Cada linha é uma candidata da fronteira**, com o ganho, a consistência, o VIF e as quatro respostas (True/False).
>   Entra a de maior ganho que passa nos quatro; depois as outras são testadas de novo.
> - **O que esperar:** o cheque eletrônico (`is_electronic_check`) fica na fronteira (ganho de cerca de 0,002) e passa nos
>   quatro critérios: melhora o modelo em quase todas as dobras, o coeficiente é claramente positivo, não repete
>   nenhuma variável e vira uma ação concreta (incentivo ao débito automático). A 5.5 confirma: os nove métodos
>   clássicos o escolhem, e a 5.5e mostra que as outras três formas de pagamento têm o mesmo risco.
> - **O preço (`MonthlyCharges`) não entra:** o ganho é pequeno e pouco consistente, ele repete o tipo de internet e não
>   traz uma ação nova.

### 5.2c Teoria: interação entre variáveis (por que `new_monthly_customer`)

Num modelo sem interação, o efeito do contrato é o mesmo para qualquer tempo de casa: as curvas de cada contrato
andam **paralelas** (no log da chance). Quando o efeito de uma variável **depende** da outra, as curvas deixam de ser
paralelas, e é preciso uma coluna de **interação**. `new_monthly_customer` (1 = até 6 meses **e** contrato mensal) é
uma interação: o risco do contrato mensal é muito maior nos primeiros meses do que depois. O gráfico abaixo é o
**gráfico de interação**: o churn por faixa de tempo de casa, uma linha por contrato.

In [ ]:
# 5.2c Gráfico de interação: churn por faixa de tempo de casa, uma linha por contrato
FAIXAS = [0, 6, 12, 24, 48, 72]
rotulos = ["0 a 6", "7 a 12", "13 a 24", "25 a 48", "49 a 72"]
faixa = pd.cut(treino["tenure"], FAIXAS, labels=rotulos, include_lowest=True)
inter = treino.groupby([faixa, "Contract"], observed=True)["Churn"].agg(["mean", "size"]).reset_index()
display(inter.pivot(index="tenure", columns="Contract", values="mean").mul(100).round(1))

fig, ax = plt.subplots(figsize=(10, 4.2))
cores_c = {"Month-to-month": DESTAQUE, "One year": COR, "Two year": CINZA}
for contrato, d in inter.groupby("Contract"):
    d = d[d["size"] >= 30]                                             # só faixas com pelo menos 30 clientes
    ax.plot(d["tenure"].astype(str), 100 * d["mean"], marker="o", color=cores_c[contrato], linewidth=2)
    ultimo = d.iloc[-1]
    ax.text(len(rotulos) - 0.9, 100 * ultimo["mean"], contrato, va="center", fontsize=9, color=cores_c[contrato])
m6 = inter[(inter["Contract"] == "Month-to-month") & (inter["tenure"] == "0 a 6")]["mean"].iloc[0]
m_fim = inter[(inter["Contract"] == "Month-to-month") & (inter["tenure"] == "49 a 72")]["mean"].iloc[0]
ax.set_xlim(-0.3, len(rotulos) + 0.3)
ax.set_xlabel("tempo de casa (meses); só faixas com 30 clientes ou mais (dois anos de 0 a 6 meses fica de fora)")
ax.set_ylabel("churn (%)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0f}%"))
fig.suptitle(f"No contrato mensal, o churn cai de {virg(100 * m6, 0)}% nos primeiros 6 meses para {virg(100 * m_fim, 0)}% depois de 4 anos;\n"
             "nos contratos anual e de dois anos, ele é baixo desde o início: as curvas não são paralelas (interação)",
             x=0.01, ha="left", fontsize=12, fontweight="bold")
rodape(fig, ["Contract", "tenure", "Churn"], fonte="gold_churn_treino (5.634 clientes)")
plt.show()

> **Como ler o gráfico de interação**
> - **Cada linha é um contrato;** o eixo horizontal é o tempo de casa. Se o efeito do contrato fosse o mesmo em todo o
>   tempo de casa, as linhas andariam paralelas.
> - **O contrato mensal despenca** nos primeiros meses e os outros quase não mudam: o risco do mensal depende do tempo
>   de casa. É isso que `new_monthly_customer` captura numa coluna só, e por isso ela foi a primeira a entrar na 5.2.
> - **Para o negócio:** o momento crítico é o começo do contrato mensal. É ali que a ação de boas-vindas e a oferta de
>   migração para contrato anual mais rendem.

In [ ]:
# 5.3 Comparação: a seleção contra a linha de base, o modelo com tudo e o contraexemplo com texto (D9)
nao_descartadas = dic.loc[dic["sugestao"].isin(["recomendada", "candidata"]), "coluna"].tolist()
auc_tudo, dp_tudo = auc_cv(nao_descartadas)
auc_regra = roc_auc_score(y, treino["IsHighRiskContract"])          # regra sem modelo: mensal e cheque eletrônico
txt_treino = texto[texto["base"] == "treino"].set_index("customerID").loc[treino["customerID"], "fl_menciona_cancelamento"]
X_texto = mat(selecionadas).assign(_texto=txt_treino.values)      # a seleção + o indicador de cancelamento no texto
notas_texto = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)), X_texto.values, y,
                              cv=CV, scoring="roc_auc")
auc_texto = notas_texto.mean()

comparacao = pd.DataFrame([
    {"modelo": "Regra sem modelo: contrato mensal e cheque eletrônico", "n_variaveis": 1, "AUC": auc_regra},
    {"modelo": "Só as 9 recomendadas", "n_variaveis": len(RECOMENDADAS), "AUC": passos.loc[0, "AUC"]},
    {"modelo": "Seleção da conferência", "n_variaveis": len(selecionadas), "AUC": passos["AUC"].iloc[-1]},
    {"modelo": "Todas as recomendadas e candidatas (com repetições)", "n_variaveis": len(nao_descartadas), "AUC": auc_tudo},
    {"modelo": "Contraexemplo: seleção + texto do feedback (vazamento, D9)", "n_variaveis": len(selecionadas) + 1, "AUC": auc_texto},
])
display(comparacao.round(4))

fig, ax = plt.subplots(figsize=(11, 4))
yy = np.arange(len(comparacao))[::-1]
cores = [CINZA, COR, COR, CINZA, DESTAQUE]
ax.barh(yy, comparacao["AUC"] - 0.5, left=0.5, color=cores, height=0.6)
for y0, (_, r) in zip(yy, comparacao.iterrows()):
    ax.text(r["AUC"] + 0.004, y0, f"AUC {virg(r['AUC'], 3)}  ·  {r['n_variaveis']} variáve{'l' if r['n_variaveis'] == 1 else 'is'}", va="center", fontsize=9, color=TEXTO)
ax.set_yticks(yy, comparacao["modelo"], fontsize=9)
ax.set_xlim(0.5, 1.17)
ax.set_xticks(np.arange(0.5, 1.01, 0.1))                       # a AUC vai até 1: sem marcas depois disso
ax.axvline(0.5, color="#999999", linewidth=1)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: virg(v, 2)))
ax.set_xlabel("AUC (Area Under the Curve) na validação cruzada; 0,5 = sorteio, 1 = perfeito")
ax.tick_params(axis="y", length=0)
ax.spines["left"].set_visible(False)
fig.suptitle(f"A seleção da conferência chega a AUC {virg(passos['AUC'].iloc[-1], 3)} com {len(selecionadas)} variáveis, praticamente o mesmo que usar todas;\n"
             f"com o texto do feedback, a AUC salta para {virg(auc_texto, 3)}: é o vazamento, e não um modelo melhor",
             x=0.01, ha="left", fontsize=12, fontweight="bold")
rodape(fig, ["Contract", "PaymentMethod", "CustomerFeedback"], fonte="gold_churn_treino e gold_churn_texto")
plt.show()

> **Como ler a comparação**
> - **Regra sem modelo:** "contrato mensal e cheque eletrônico" já separa melhor que o sorteio. É a régua mínima
>   (seção 10 do 03).
> - **Só as recomendadas × seleção da conferência:** o quanto as candidatas acrescentam.
> - **Todas as recomendadas e candidatas:** usa tudo, inclusive as alternativas que dizem a mesma coisa. Se a AUC fica
>   parecida com a da seleção, as variáveis extras **não acrescentam**: só deixam o modelo mais difícil de explicar e
>   mais instável.
> - **Contraexemplo com texto:** o salto da AUC é o vazamento da seção 7.3 do 03. É o número que **não** usamos como
>   resultado (D9), e que mostramos na apresentação para explicar por quê.

### 5.4 Teoria: conferir a repetição entre as selecionadas (VIF)

O **VIF** (*Variance Inflation Factor*, fator de inflação da variância) mede, para cada coluna do modelo, o quanto ela
é explicada pelas **outras** colunas. A conta: faço uma regressão da coluna contra todas as outras e pego o R²
(a fração explicada); VIF = 1 ÷ (1 − R²).

A **tolerância** é o outro lado da mesma conta: tolerância = 1 − R² = 1 ÷ VIF, a parte da coluna que as outras
**não** explicam.

| VIF | Tolerância | Leitura |
|---|---|---|
| 1 | 1 | A coluna não é explicada pelas outras |
| até 5 | 0,2 ou mais | Repetição aceitável |
| 5 a 10 | 0,1 a 0,2 | Atenção: os coeficientes ficam instáveis |
| acima de 10 | abaixo de 0,1 | Repetição demais: tirar uma das variáveis repetidas |

Na logística, VIF alto não piora a AUC, mas deixa os **coeficientes** sem sentido (o efeito se divide entre as
colunas repetidas), e é pelos coeficientes que explicamos o modelo ao negócio.

In [ ]:
# 5.4 VIF e tolerância das colunas da seleção da conferência (depois das dummies); a função vif_tabela está na 5.1
vif = vif_tabela(selecionadas)
display(vif.round(3))
print(f"VIF máximo: {virg(vif['VIF'].max(), 2)} ({vif.iloc[0]['coluna_do_modelo']}); "
      f"tolerância mínima: {virg(vif['tolerancia'].min(), 2)}")

> **Como ler o VIF**
> - **Cada linha é uma coluna do modelo** (depois das dummies: `Contract` vira `Contract_One year` e `Contract_Two year`).
> - O que esperar: `tenure` e as interações que usam tempo de casa ou contrato (por exemplo, `new_monthly_customer`)
>   costumam ter VIF maior, porque são parcialmente explicadas por `tenure` e `Contract`. Até 5, é aceitável.

### 5.5 Conferência com os métodos clássicos: *stepwise* por AIC, BIC e p-valor

A seleção da 5.2 usa a AUC na validação cruzada e grupos de alternativas. Os métodos clássicos de regressão escolhem
as variáveis de outro jeito. Se os dois caminhos chegarem às mesmas variáveis, a escolha fica mais sólida; onde
discordarem, vale entender por quê.

#### Teoria: o modelo nulo e a log-verossimilhança

- **Modelo nulo:** só o intercepto, sem nenhuma variável. Ele dá a mesma probabilidade para todos os clientes, a taxa
  de churn da base (26,7%). É o "chute da média" da regressão linear, adaptado para o Sim/Não.
- **Log-verossimilhança:** o quanto as probabilidades do modelo combinam com o que aconteceu. Para cada cliente, pego
  a probabilidade que o modelo deu ao que de fato ocorreu (saiu ou ficou) e somo os logaritmos. Quanto mais perto de
  zero, melhor o ajuste. Na logística, ela faz o papel da soma dos erros ao quadrado da regressão linear.

#### Teoria: AIC, BIC e p-valor como critério de entrada

O **AIC** (*Akaike Information Criterion*, critério de informação de Akaike) junta ajuste e tamanho:

> **AIC = −2 × log-verossimilhança + 2 × nº de parâmetros** (menor é melhor)

Uma variável só compensa se melhorar o ajuste mais do que o "preço" dos parâmetros que ela acrescenta. O critério
**por p-valor** é a mesma conta com um preço maior: **3,84 por parâmetro, em vez de 2**. O 3,84 é o valor da
distribuição qui-quadrado com 1 grau de liberdade no nível de 5%. Pagar 3,84 equivale a exigir que o teste da razão de
verossimilhança da variável dê p-valor abaixo de 0,05. Mais exigente, ele escolhe menos variáveis. É o que faz no R
o `step(modelo, k = 3.841459)`.

O **BIC** (*Bayesian Information Criterion*, critério de informação bayesiano) é a mesma conta com o preço
**ln(n)** por parâmetro, em que n é o número de clientes: ln(5.634) ≈ 8,64. É o mais exigente dos três, e o preço
cresce com o tamanho da base.

| Critério | Preço por parâmetro | No R |
|---|---|---|
| AIC | 2 | `step(modelo)` |
| p-valor de 5% | 3,84 | `step(modelo, k = 3.841459)` |
| BIC | ln(5.634) ≈ 8,64 | `step(modelo, k = log(nrow(base)))` |

#### Teoria: as três direções

| Direção | Começa em | A cada passo | Para quando |
|---|---|---|---|
| ***Forward*** (para frente) | Modelo nulo | Entra a variável que mais reduz o critério | Nenhuma entrada reduz o critério |
| ***Backward*** (para trás) | Modelo com todas as 30 variáveis | Sai a variável cuja saída mais reduz o critério | Nenhuma saída reduz o critério |
| ***Stepwise*** (nos dois sentidos) | Modelo nulo | Entra ou sai, o que mais reduzir o critério | Nenhuma entrada ou saída reduz o critério |

**Duas diferenças em relação à seleção da 5.2:**
1. **O ajuste é medido no próprio treino** (com a penalidade dos parâmetros), e não em clientes que o modelo não viu.
2. **Os métodos não conhecem os grupos de alternativas:** podem colocar no modelo duas variáveis que dizem a mesma
   coisa, se a combinação das duas melhorar um pouco o ajuste.

#### O que comparar entre os métodos

| Medida | Pergunta |
|---|---|
| Nº de variáveis | O modelo é simples de explicar? |
| AIC | O ajuste no treino, descontado o tamanho |
| AUC na validação cruzada | Ordena bem quem sai, em clientes que o modelo não viu? |
| VIF máximo | Há variáveis repetindo umas às outras (5.4)? |
| Sinais invertidos | Algum coeficiente diz o contrário do que o dado mostra sozinho? |

**Sinal invertido:** sozinha, cada coluna tem uma direção clara (por exemplo, quem tem fibra sai mais que quem tem
DSL). Se, no modelo, o coeficiente dela aponta para o lado contrário, há duas explicações possíveis: outra variável do
modelo já explica aquele efeito (confusão, que pode ser legítima), ou há repetição demais entre as variáveis
(multicolinearidade), e os coeficientes deixaram de fazer sentido. VIF alto junto com sinal invertido é o sintoma
clássico do segundo caso.

In [ ]:
# 5.5a Os nove métodos clássicos (forward, backward e stepwise; AIC, p-valor e BIC) contra a seleção da conferência
import time
_ajustes = {}
def log_verossimilhanca(colunas: list) -> tuple:
    """Log-verossimilhança e nº de parâmetros (colunas + intercepto) da logística sem penalização, ajustada no treino."""
    chave = tuple(sorted(colunas))
    if chave not in _ajustes:
        if not colunas:                                                 # modelo nulo: só o intercepto
            p = y.mean()
            _ajustes[chave] = (float((y * np.log(p) + (1 - y) * np.log(1 - p)).sum()), 1)
        else:
            X = mat(colunas)
            _, p, _, _ = ajustar_logistica(X)
            _ajustes[chave] = (float((y * np.log(p) + (1 - y) * np.log(1 - p)).sum()), X.shape[1] + 1)
    return _ajustes[chave]

def criterio(colunas: list, preco: float) -> float:
    ll, n_par = log_verossimilhanca(colunas)
    return -2 * ll + preco * n_par                                      # preço 2 = AIC; 3,84 = p-valor de 5%; ln(n) = BIC

def stepwise(preco: float, direcao: str) -> list:
    atual_sel = [] if direcao in ("forward", "stepwise") else list(TODAS)
    atual = criterio(atual_sel, preco)
    while True:
        opcoes = []
        if direcao in ("forward", "stepwise"):
            opcoes += [(criterio(atual_sel + [v], preco), atual_sel + [v]) for v in TODAS if v not in atual_sel]
        if direcao in ("backward", "stepwise"):
            opcoes += [(criterio([c for c in atual_sel if c != v], preco), [c for c in atual_sel if c != v]) for v in atual_sel]
        if not opcoes:
            break
        melhor, nova_sel = min(opcoes, key=lambda o: o[0])
        if melhor >= atual - 1e-9:
            break
        atual, atual_sel = melhor, nova_sel
    return atual_sel

METODOS = {}
inicio = time.time()
for preco, nome_criterio in [(2, "AIC"), (3.841459, "p-valor"), (np.log(len(y)), "BIC")]:
    for direcao in ["forward", "backward", "stepwise"]:
        METODOS[f"{direcao} por {nome_criterio}"] = stepwise(preco, direcao)
        print(f"  {direcao} por {nome_criterio}: {len(METODOS[f'{direcao} por {nome_criterio}'])} variáveis "
              f"({virg(time.time() - inicio, 0)} s até aqui)", flush=True)
METODOS["conferência (AUC + grupos)"] = list(selecionadas)
METODOS["todas as 30"] = list(TODAS)

def sinais_invertidos(colunas: list) -> list:
    """Colunas cujo coeficiente no modelo aponta para o lado contrário da relação simples com o churn."""
    X = mat(colunas)
    coef = coeficientes(colunas)["coef"]
    sozinha = [np.sign(np.corrcoef(X[c], y)[0, 1]) for c in X.columns]
    return [c for c, b, s_ in zip(X.columns, coef, sozinha) if np.sign(b) != s_ and abs(b) > 1e-6]

ll_nulo = log_verossimilhanca([])[0]
comparacao_metodos = pd.DataFrame([{
    "metodo": nome, "n_variaveis": len(cols), "AIC": criterio(cols, 2), "BIC": criterio(cols, np.log(len(y))),
    "pseudo_R2": 1 - log_verossimilhanca(cols)[0] / ll_nulo,          # McFadden: quanto do "erro" do nulo o modelo tira
    "AUC_validacao_cruzada": auc_cv(cols)[0], "VIF_maximo": vif_tabela(cols)["VIF"].max(),
    "sinais_invertidos": ", ".join(sinais_invertidos(cols)) or "nenhum",
} for nome, cols in METODOS.items()])
display(comparacao_metodos.round(3))
print(f"Modelo nulo (só o intercepto): AIC {virg(criterio([], 2), 0)}")

> **Como ler a comparação dos métodos**
> - **Os nove métodos clássicos chegam a modelos parecidos entre si** (10 a 14 variáveis), com AUC praticamente
>   igual. A direção (*forward*, *backward* ou *stepwise*) muda pouco; o critério muda mais: o BIC, mais exigente,
>   escolhe menos variáveis.
> - **AIC e BIC:** os métodos clássicos têm AIC e BIC menores que a seleção da conferência. É esperado: são os critérios que
>   eles minimizam, no próprio treino.
> - **Pseudo-R² (de McFadden):** 1 − log-verossimilhança do modelo ÷ a do nulo. Lembra o R² da regressão linear,
>   mas na logística valores de 0,2 a 0,4 já indicam um bom ajuste.
> - **AUC na validação cruzada:** a diferença entre os métodos e a seleção da conferência é de cerca de 0,002, abaixo do
>   critério de ganho mínimo da 5.2.
> - **VIF máximo e sinais invertidos:** é aqui que os caminhos se separam. Oito dos nove métodos colocam as duas
>   variáveis de preço junto com o tipo de internet, e o VIF passa de 80. A exceção é o *stepwise* por BIC, que tira
>   `InternetService` e deixa o preço no lugar dele: VIF baixo e sinais coerentes, uma alternativa válida. A tabela da 5.5b mostra quais variáveis
>   cada método escolheu, e o gráfico da 5.5c mostra o que acontece com os coeficientes.

In [ ]:
# 5.5b Quais variáveis cada método escolheu
nomes_metodos = [m for m in METODOS if m != "todas as 30"]
todas_escolhidas = sorted({v for m in nomes_metodos for v in METODOS[m]},
                          key=lambda v: (-sum(v in METODOS[m] for m in nomes_metodos), v))
presenca = pd.DataFrame({m: [v in METODOS[m] for v in todas_escolhidas] for m in nomes_metodos}, index=todas_escolhidas)
GRUPO_DE = {v: g for g, membros in GRUPOS.items() for v in membros}
precos_juntos = [m for m in nomes_metodos if sum(v in METODOS[m] for v in GRUPOS["preço"]) == 2]
classicos = [m for m in nomes_metodos if m != "conferência (AUC + grupos)"]
em_todos = [v for v in selecionadas if all(v in METODOS[m] for m in classicos)]

fig, ax = plt.subplots(figsize=(13, 0.36 * len(todas_escolhidas) + 2.4))
for j, m in enumerate(nomes_metodos):
    for i, v in enumerate(todas_escolhidas):
        if presenca.loc[v, m]:
            cor = DESTAQUE if GRUPO_DE.get(v) == "preço" else COR
            ax.scatter(j, i, s=140, color=cor, zorder=3)
ax.axvline(len(nomes_metodos) - 1.5, color="#999999", linewidth=1, linestyle="--")
ax.set_xticks(range(len(nomes_metodos)), [textwrap.fill(m, 12) for m in nomes_metodos], fontsize=8)
ax.set_yticks(range(len(todas_escolhidas)), todas_escolhidas, fontsize=9)
ax.set_ylim(len(todas_escolhidas) - 0.5, -0.5)
ax.set_xlim(-0.6, len(nomes_metodos) - 0.4)
ax.grid(color="#eeeeee", zorder=0)
ax.tick_params(length=0)
ax.spines[["left", "bottom"]].set_visible(False)
ax.xaxis.tick_top()
com_internet = [m for m in precos_juntos if "InternetService" in METODOS[m]]
fig.suptitle(f"{len(em_todos)} das nossas {len(selecionadas)} variáveis aparecem nos {len(classicos)} métodos clássicos; a diferença: {len(precos_juntos)} deles usam as duas\n"
             f"variáveis de preço juntas (em laranja), e {len(com_internet)} desses ainda somam o tipo de internet, que o preço já repete",
             x=0.01, ha="left", fontsize=12, fontweight="bold")
rodape(fig, ["MonthlyCharges", "Contract", "InternetService", "tenure"], fonte="gold_churn_treino (5.634 clientes)")
plt.show()
print("Em todos os métodos clássicos e na conferência:", ", ".join(em_todos))
print("Só na conferência:", ", ".join(v for v in selecionadas if not any(v in METODOS[m] for m in classicos)) or "nenhuma")

In [ ]:
# 5.5c Coeficientes com intervalo de 95%: conferência × stepwise por p-valor
# a função coeficientes está na 5.1
CLASSICO = "stepwise por p-valor"
coef_nossa, coef_class = coeficientes(METODOS["conferência (AUC + grupos)"]), coeficientes(METODOS[CLASSICO])
linhas = list(dict.fromkeys(list(coef_class.index) + list(coef_nossa.index)))
fibra = "InternetService_Fiber optic"
display(pd.concat({"conferência": coef_nossa, CLASSICO: coef_class}, axis=1).reindex(linhas).round(2))

fig, ax = plt.subplots(figsize=(11, 0.42 * len(linhas) + 2))
for k, (nome, tab, cor, desloc) in enumerate([("conferência (AUC + grupos)", coef_nossa, COR, -0.17),
                                                (CLASSICO, coef_class, DESTAQUE, 0.17)]):
    for i, c in enumerate(linhas):
        if c in tab.index:
            r = tab.loc[c]
            ax.plot([r["inf"], r["sup"]], [i + desloc] * 2, color=cor, linewidth=2)
            ax.scatter(r["coef"], i + desloc, color=cor, s=30, zorder=3, label=nome if i == min(j for j, x in enumerate(linhas) if x in tab.index) else None)
ax.axvline(0, color="#666666", linewidth=1)
ax.set_yticks(range(len(linhas)), linhas, fontsize=9)
ax.set_ylim(len(linhas) - 0.5, -0.5)
ax.set_xlabel("coeficiente padronizado (efeito de 1 desvio-padrão da coluna no log da chance de sair), com intervalo de 95%")
ax.legend(frameon=False, loc="lower right", fontsize=9)
ax.tick_params(axis="y", length=0)
ax.spines["left"].set_visible(False)
sinal = "reduzir" if coef_class.loc[fibra, "coef"] < 0 else "aumentar"
fig.suptitle(f"No {CLASSICO}, a fibra passa a {sinal} o risco e os coeficientes de preço ficam enormes e opostos:\n"
             "é a multicolinearidade; na conferência, os sinais seguem o que os dados mostram",
             x=0.01, ha="left", fontsize=12, fontweight="bold")
rodape(fig, ["InternetService", "MonthlyCharges", "Contract", "tenure"], fonte="gold_churn_treino (5.634 clientes)")
plt.show()

In [ ]:
# 5.5d Correlação entre as colunas do stepwise por p-valor: de onde vem a repetição
X_class = mat(METODOS[CLASSICO])
mostrar_corr = [c for c in X_class.columns if c in ("MonthlyCharges", "mensalidade_relativa_pacote", "tenure", fibra,
                "InternetService_No", "OnlineSecurity_bin", "TechSupport_bin", "OnlineBackup_bin", "Contract_Two year")]
corr = X_class[mostrar_corr].corr()
fig, ax = plt.subplots(figsize=(9, 7))
mascara = np.triu(np.ones_like(corr, dtype=bool), k=1)
dados_corr = corr.mask(mascara)
im = ax.imshow(dados_corr, cmap="RdBu_r", vmin=-1, vmax=1)
for i in range(len(corr)):
    for j in range(i + 1):
        v = corr.iloc[i, j]
        ax.text(j, i, virg(v, 2), ha="center", va="center", fontsize=8.5, color="white" if abs(v) > 0.6 else TEXTO)
ax.set_xticks(range(len(corr)), [str(k + 1) for k in range(len(corr))], fontsize=9)          # números, para não inclinar os nomes
ax.set_yticks(range(len(corr)), [f"{k + 1}  {c}" for k, c in enumerate(corr.index)], fontsize=8.5)
ax.spines[:].set_visible(False)
ax.tick_params(length=0)
fig.colorbar(im, ax=ax, shrink=0.7, label="correlação de Pearson")
r_fibra = corr.loc["MonthlyCharges", fibra]
vif_class = vif_tabela(METODOS[CLASSICO]).set_index("coluna_do_modelo")["VIF"]
fig.suptitle(f"A mensalidade anda junto com o tipo de internet (correlação de {virg(r_fibra, 2)} com fibra): somada à mensalidade\n"
             f"relativa e aos serviços, ela é quase toda explicada pelas outras colunas (VIF {virg(vif_class['MonthlyCharges'], 0)})",
             x=0.01, ha="left", fontsize=12, fontweight="bold")
rodape(fig, ["MonthlyCharges", "InternetService", "tenure", "Contract"], fonte="gold_churn_treino (5.634 clientes)")
plt.show()

> **Como ler os gráficos da 5.5**
> - **5.5b (quais variáveis):** cada coluna é um método, cada ponto uma variável escolhida. Laranja são as duas
>   variáveis de preço (`MonthlyCharges` e `mensalidade_relativa_pacote`), que formam um grupo de alternativas na 5.1.
> - **5.5c (coeficientes):** cada linha é uma coluna do modelo; o ponto é o coeficiente padronizado e a barra, o
>   intervalo de 95%. À direita do zero, aumenta o risco; à esquerda, reduz. Barras que cruzam o zero indicam efeito
>   incerto. Repare na fibra e nas duas variáveis de preço no modelo clássico.
> - **5.5d (correlação):** os números das colunas repetem os das linhas. A correlação dois a dois explica parte da repetição. O VIF mostra o resto: a mensalidade é
>   explicada pela **combinação** do tipo de internet, da mensalidade relativa e dos serviços, mesmo sem uma
>   correlação dois a dois perto de 1.
>
> **Resposta: por que a conferência fica com esta seleção**
> - **Os métodos clássicos confirmam o núcleo:** contrato, tempo de casa, fatura digital, idoso, cliente novo no
>   contrato mensal e cheque eletrônico aparecem em todos os nove. O tipo de internet aparece em oito (o *stepwise*
>   por BIC troca pelo preço). Segurança online e suporte técnico aparecem nos *forward* e *stepwise*; nos *backward*,
>   entram no lugar variações como `n_protecao` e a proteção de aparelho, que dizem quase o mesmo.
> - **Os *backward* são instáveis aqui:** eles partem do modelo com todas as 30, em que algumas colunas repetem outras
>   exatamente (VIF infinito). Nessa situação o ajuste tem várias soluções equivalentes, e pequenas diferenças de
>   arredondamento entre computadores mudam o caminho (uma variável a mais ou a menos). É mais um sintoma da
>   multicolinearidade, e um motivo para começar pela triagem (04) em vez de jogar tudo no modelo.
> - **A diferença é o preço:** sem os grupos de alternativas, os métodos clássicos colocam as duas variáveis de preço
>   juntas e, em oito dos nove, também o tipo de internet. O ajuste melhora um pouco no treino, mas o VIF explode, e o coeficiente da fibra muda de sinal: o modelo
>   passaria a dizer que fibra reduz o risco, quando 42% dos clientes de fibra saem, contra 19% no DSL (03).
> - **Para explicar ao negócio, os coeficientes precisam fazer sentido.** A AUC é praticamente a mesma (diferença
>   de cerca de 0,002), então ficamos com a seleção de VIF baixo e sinais coerentes (D19).
> - **A alternativa do BIC:** o *stepwise* por BIC (10 variáveis) troca o tipo de internet pelas duas variáveis de
>   preço, com VIF baixo, sinais coerentes e AUC igual. É uma alternativa válida; ficamos com o tipo de internet porque
>   ele vira uma ação clara (a oferta para clientes de fibra), mas a modelagem pode comparar as duas.
> - **`Partner` e `Dependents`:** sozinhos, reduzem o churn (20% contra 33% com parceiro(a); 15% contra 32% com
>   dependentes), mas no modelo os coeficientes ficam perto de zero, com o intervalo cruzando o zero (5.5c), e o de
>   `Partner` fica até levemente positivo. É confusão, e não repetição (VIF baixo): quem tem família tem mais tempo de
>   casa, e o modelo já conta o tempo de casa. Na modelagem, vale testar o modelo sem os dois.

#### Teoria: agrupar categorias pelo modelo

Quando o coeficiente de uma categoria tem intervalo de 95% que **inclui o zero**, o modelo não consegue distinguir
essa categoria da referência: na presença das outras variáveis, as duas têm o mesmo risco. Nesse caso, faz sentido
**juntar** as categorias, com menos colunas e o mesmo poder. O agrupamento também pode vir do negócio ou de categorias
com a mesma taxa de churn.

**No nosso caso:** `PaymentMethod` tem 4 categorias. Em vez dela, a seleção usa `is_electronic_check` (1 = cheque
eletrônico; 0 = as outras três juntas). A célula 5.5e confere se esse agrupamento se justifica: coloco
`PaymentMethod` no lugar de `is_electronic_check` e olho os intervalos das categorias.

In [ ]:
# 5.5e Agrupar as formas de pagamento: as três que não são cheque eletrônico têm o mesmo risco?
com_pagamento = [c for c in selecionadas if c != "is_electronic_check"] + ["PaymentMethod"]
coef_pag = coeficientes(com_pagamento).filter(like="PaymentMethod_", axis=0)
taxa_pag = treino.groupby("PaymentMethod")["Churn"].agg(clientes="size", churn="mean")
display(coef_pag.round(2))
display((taxa_pag.assign(churn_pct=lambda d: 100 * d["churn"]).drop(columns="churn")).round(1))
ref = CATEGORIAS["PaymentMethod"][0]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.4), gridspec_kw={"width_ratios": [1.2, 1]})
cats = [ref] + [c.replace("PaymentMethod_", "") for c in coef_pag.index]
yy = np.arange(len(cats))[::-1]
for y0, c in zip(yy, cats):
    if c == ref:
        ax1.scatter(0, y0, color=CINZA, s=40, zorder=3)
        ax1.text(0.03, y0, "referência", va="center", fontsize=8.5, color="#666666")
        continue
    r = coef_pag.loc[f"PaymentMethod_{c}"]
    cor = DESTAQUE if r["inf"] > 0 else COR
    ax1.plot([r["inf"], r["sup"]], [y0, y0], color=cor, linewidth=2.5)
    ax1.scatter(r["coef"], y0, color=cor, s=40, zorder=3)
ax1.axvline(0, color="#666666", linewidth=1)
ax1.set_yticks(yy, cats, fontsize=9)
ax1.set_xlabel("coeficiente padronizado, com intervalo de 95%")
ax1.set_title("No modelo (com as outras 10 variáveis)", loc="left", fontsize=10.5)
ax1.tick_params(axis="y", length=0)
ax1.spines["left"].set_visible(False)
v = taxa_pag.reindex(cats)["churn"] * 100
ax2.barh(yy, v, color=[DESTAQUE if c == "Electronic check" else CINZA for c in cats], height=0.6)
for y0, x in zip(yy, v):
    ax2.text(x + 0.8, y0, f"{virg(x, 1)}%", va="center", fontsize=9, color=TEXTO)
ax2.set_yticks(yy, [""] * len(cats))
ax2.set_xlim(0, v.max() * 1.25)
for a in (ax1, ax2):
    a.set_ylim(-0.5, len(cats) - 0.5)                                # mesma altura nos dois painéis
ax2.set_xticks([])
ax2.set_title("Churn de cada forma de pagamento", loc="left", fontsize=10.5)
ax2.spines[["bottom", "left"]].set_visible(False)
ax2.tick_params(axis="y", length=0)
juntas = [c.replace("PaymentMethod_", "") for c, r in coef_pag.iterrows() if r["inf"] <= 0 <= r["sup"]]
fig.suptitle(f"No modelo, {' e '.join(juntas)} não se distinguem de {ref} (intervalos cruzam o zero):\n"
             "as três formas podem ser juntas, e só o cheque eletrônico fica à parte, como em is_electronic_check",
             x=0.01, ha="left", fontsize=12, fontweight="bold")
rodape(fig, ["PaymentMethod", "Churn"], fonte="gold_churn_treino (5.634 clientes)")
plt.show()

> **Como ler e resposta (5.5e)**
> - **À esquerda,** o efeito de cada forma de pagamento em relação à referência (transferência automática), já
>   descontadas as outras variáveis. Cartão de crédito e boleto pelo correio têm o intervalo cruzando o zero: na
>   presença das outras variáveis, têm o mesmo risco da transferência. Só o cheque eletrônico fica do lado do risco.
> - **À direita,** a taxa de churn de cada forma confirma: cerca de 16% a 19% nas três, contra 45% no cheque eletrônico.
> - **Resposta:** o agrupamento em `is_electronic_check` (uma coluna, em vez de três) se justifica pelo modelo e pelos
>   dados. É o mesmo raciocínio de agrupar categorias com a mesma média de Y.

In [ ]:
# 5.6 Grava a seleção da conferência (gold_churn_selecao), usada pela segmentação do 07 como plano B
selecao = dic.set_index("coluna").loc[selecionadas, ["origem", "tipo_derivada", "formula", "IV", "sugestao", "motivo"]].reset_index()
selecao.insert(0, "ordem_de_entrada", range(1, len(selecao) + 1))
na_fronteira = set(passos.loc[passos["grupo"].str.contains("fronteira"), "entrou"])
selecao["como_entrou"] = ["ponto de partida (recomendada)" if c in RECOMENDADAS else
                          "decisão na fronteira (5.2b)" if c in na_fronteira else "seleção para frente (5.2)" for c in selecao["coluna"]]
(spark.createDataFrame(selecao).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{FQ}.gold_churn_selecao"))
spark.sql(f"COMMENT ON TABLE {FQ}.gold_churn_selecao IS 'Seleção da conferência no notebook 05 (logística, seleção para frente por AUC na validação cruzada). Conferência da seleção de variáveis com os métodos da aula'")
display(selecao[["ordem_de_entrada", "coluna", "como_entrou", "IV", "motivo"]])
print(f"{FQ}.gold_churn_selecao: {len(selecao)} variáveis")

## 6. As duas seleções se encontram?

> **M1 — Que variáveis entram no modelo, e por quê?** (parte 3: o encontro)

In [ ]:
# 6.1 As variáveis da conferência (seção 5) estão entre as 41 colunas do modelo?
finais = X_treino_limpo.columns.tolist()
EQUIVALENTE = {"is_electronic_check": "PaymentMethod_Electronic check"}   # duplicata exata (3.11)
linhas = []
for v in selecionadas:
    if v in CATEGORIAS:                                  # categórica: entra pelas dummies
        cols = [c for c in finais if c.startswith(f"{v}_")]
    else:
        cols = [c for c in finais if c == v]
    como = (("a dummy" if len(cols) == 1 else "as dummies") + " da mesma variável" if v in CATEGORIAS else "a mesma coluna") if cols else ""
    if not cols and v in EQUIVALENTE and EQUIVALENTE[v] in finais:
        cols, como = [EQUIVALENTE[v]], "coluna equivalente (duplicata exata, 3.11)"
    linhas.append({"variavel_da_conferencia": v, "colunas_nas_41": ", ".join(cols) or "—",
                   "como": como or "não está nas 41", "esta_nas_41": bool(cols)})
encontro = pd.DataFrame(linhas)
display(encontro)
usadas = {c for t in encontro["colunas_nas_41"] for c in t.split(", ") if c != "—"}
print(f"{encontro['esta_nas_41'].sum()} de {len(encontro)} variáveis da conferência estão nas 41 colunas do modelo; "
      f"elas ocupam {len(usadas)} das 41 colunas. As outras {len(finais) - len(usadas)} são as que os testes da seção 3 "
      "mantiveram e que a regularização do 06 pode encolher.")

> **Como ler e resposta (M1)**
> - **Cada linha é uma variável da conferência** (seção 5) e a coluna (ou colunas, no caso das categóricas) que ela
>   ocupa nas 41 do modelo. `is_electronic_check` entra pela coluna equivalente `PaymentMethod_Electronic check`.
> - **Leitura:** os dois caminhos concordam no que importa: contrato, tempo de casa, internet, forma de pagamento e os
>   serviços de proteção estão nas duas seleções. As 41 são mais largas porque a seleção final retira só o que é
>   **comprovadamente** repetido ou sem associação, e deixa para a regularização do 06 o ajuste fino.
> - **M1 — Que variáveis entram no modelo, e por quê?** As 41 colunas da 3.12: saem as 6 categóricas redundantes
>   (ficam as versões sim/não) e 12 variáveis por duplicata exata, versão repetida do tempo de casa ou falta de
>   associação com o churn após a correção para múltiplos testes (`gender`). A conferência com os métodos da aula
>   escolhe um subconjunto delas.

## Checklist final do 05

| # | Decisão | Status | Onde neste notebook |
|---|---|---|---|
| D14 | A gold está como a modelagem espera | ☑ Conferido | 1.2: teste de carga |
| D9 | Texto fora da ABT; contraexemplo com texto | ☑ Conferido | 1.2 e 5.3 |
| D13 | Inferência com 1.409 clientes | ☑ Conferido | 1.2 |
| D10 | Serviços como sim/não | ☑ As 6 categóricas de 3 níveis saem; ficam as versões `_bin` | 2.1 |
| D15 | `gender` fora | ☑ Sai: sem associação com o churn após a correção FDR | 3.4 e 3.11 |
| D19 | Seleção de variáveis | ☑ 41 colunas (seções 2 e 3); a conferência com os métodos da aula cabe dentro delas | 3.12 e 6.1 |

**Tabelas gravadas:** `gold_churn_split` (quem é treino e quem é teste, com a ordem), `gold_churn_variaveis_modelo`
(as listas de variáveis) e `gold_churn_selecao` (a conferência da seção 5).

**Por que gravar a separação:** a separação depende da ordem das linhas. Com o treino ordenado por `customerID` (1.1)
e a separação gravada, o 06 reconstrói **exatamente** as mesmas bases, em qualquer conta e em qualquer dia.

**Próximo passo:** 06 (modelagem).